# The Dual Impact of Solar Power Penetration on the Turkish Electricity Market

## Full Data Pipeline

This notebook implements the complete empirical analysis pipeline for studying the dual effect of solar power penetration on the Turkish electricity market (Day-Ahead and Balancing Markets). 

### Pipeline Overview
| Phase | Description | Method |
|-------|-------------|--------|
| 1 | Data Ingestion & Pre-processing | Temporal indexing, integrity checks |
| 2 | Multi-Model Estimation | SARIMAX, Logistic Regression |
| 3 | Parameter Justification | ACF/PACF diagnostics, Grid Search |
| 4 | Robust Estimation | Optimized SARIMAX with tuned parameters |
| 5 | Advanced Visualization | Market profiles, coefficient plots |
| 6 | Non-Linear Analysis | XGBoost PDP & Flexibility Trap; out-of-sample CV (6.4, Revision R2.3) |
| 7 | Robustness Checks | Quantile Regression (Q10/Q50/Q90); convergence diagnostics & clustered SE (7.2–7.4, Revision R2.5) |
| 8 | GAM Robustness | Cubic Spline non-linear regime analysis; Hansen threshold (8.2, Revision R2.4) |
| 9-10 | Appendices & Framework | Appendices A–J generation (incl. Botaş robustness, Appendix I; reviewer robustness, Appendix J) |
| 11 | Inflation Robustness | FE-OLS on a-FRR/MCP ratio & cap-normalised prices; timing/confounding (11.2, R2.1) & sub-period (11.3, R2.2) |

In [ ]:
# =============================================================================
# GLOBAL CONFIGURATION: OUTPUT PATHS
# =============================================================================

import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Global figure export resolution.
# Renewable Energy (Elsevier/EM) technical checklist asks for high-resolution
# figures (above 650 dpi); all savefig calls below use this single knob.
FIG_DPI = 650

# [Revision] CSV line endings: the revision cells write their CSVs with LF on every platform.
# (The repository held a CRLF/LF mix after Windows and Linux runs, which produced whole-file
# diffs with unchanged numbers.) pandas >= 1.5 calls the keyword 'lineterminator'.
import inspect
import pandas as pd
_LT_KEY = 'lineterminator' if 'lineterminator' in inspect.signature(pd.DataFrame.to_csv).parameters else 'line_terminator'
CSV_KW = {_LT_KEY: '\n'}


# Auto-detect repo root: look for the results/ directory
if os.path.isdir("results"):
    RESULTS_BASE = "results"
elif os.path.isdir(os.path.join("..", "results")):
    RESULTS_BASE = os.path.join("..", "results")
else:
    RESULTS_BASE = "results"

os.makedirs(RESULTS_BASE, exist_ok=True)
print(f">> Results base directory: {os.path.abspath(RESULTS_BASE)}")

# Data directory Configuration
if os.path.isdir("data"):
    DATA_DIR = "data"
elif os.path.isdir(os.path.join("..", "data")):
    DATA_DIR = os.path.join("..", "data")
else:
    DATA_DIR = "."

DATA_FILE = os.path.join(DATA_DIR, "Model_Data.xlsx")
print(f">> Data file path: {os.path.abspath(DATA_FILE)}")

# Global Academic Plot Styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# [Revision_Editor_E6] no background colour in figures: white figure, axes and saved-file background
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'})

# -----------------------------------------------------------------------------
# FORCE_FIGURES = True  -> asagidaki 7 cache'li PNG bu hucrede silinir; Faz 6.1,
#                          6.2, 8 ve Appendix I (XGBoost + GAM, random_state=42) ayni
#                          kosuda yeniden hesaplanip figurleri FIG_DPI'da uretir.
#                          Faz 5 ve Faz 9 zaten her kosuda yeniden cizilir.
# FORCE_FIGURES = False -> mevcut figurler korunur.
# Tazeleme bittikten sonra False'a cekin.
# -----------------------------------------------------------------------------
FORCE_FIGURES = False

_figure_cache_targets = [
    ("Phase_6", "Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png"),
    ("Phase_6", "Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png"),
    ("Phase_6", "Phase_6_Figure_8_Surplus_Probability_and_Distribution.png"),
    ("Phase_6", "Phase_6_Figure_9_Flexibility_Trap.png"),
    ("Phase_8", "Phase_8_Figure_10_GAM_Flexibility_Stress.png"),
    # [Revision_Review_Referee_2.7 / MÖK-4] Appendix I figures are cached too; without these two
    # entries a FORCE_FIGURES run left Supplementary Figures I.1 / I.2 with their old axis labels.
    ("Phase_10", "Appendix_I_Figure_I1_Flexibility_Trap_Botas.png"),
    ("Phase_10", "Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png"),
]
if FORCE_FIGURES:
    print(">> FORCE_FIGURES: figur cache'leri temizleniyor (650 dpi ile yeniden uretilecekler).")
    for _sub, _name in _figure_cache_targets:
        _p = os.path.join(RESULTS_BASE, _sub, _name)
        if os.path.exists(_p):
            os.remove(_p); print("   Silindi:", _p)
        else:
            print("   Zaten yok:", _p)


## Phase 1: Data Ingestion and Pre-Processing
Reads the raw market dataset (`Model_Data.xlsx`), establishes the temporal index, and performs initial integrity checks.

In [ ]:
# =============================================================================
# PHASE 1: DATA INGESTION AND PRE-PROCESSING
# =============================================================================
import pandas as pd
import sys

def ingest_and_structure_data(file_path: str, date_col: str = 'Date_Hour') -> pd.DataFrame:
    if not os.path.exists(file_path):
        print(f"!! ERROR: File not found at {file_path}")
        return None

    try:
        print(f">> Ingesting data from: {file_path}")
        df = pd.read_excel(file_path)

        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col])
            df.set_index(date_col, inplace=True)
            df.sort_index(inplace=True)
        else:
            print(f"!! WARNING: '{date_col}' not found. Time-series logic may fail.")
        
        return df

    except Exception as e:
        print(f"!! FATAL ERROR during data ingestion: {e}")
        return None

def generate_data_audit_report(df: pd.DataFrame):
    print("\n" + "="*60 + "\nDATASET AUDIT REPORT\n" + "="*60)
    print(f"Temporal Range : {df.index.min()} to {df.index.max()}")
    print(f"Total Observations : {len(df)}\n" + "-" * 60)
    print(f"{'Variable Name':<30} | {'Type':<10} | {'Missing Values'}\n" + "-" * 60)
    
    for col in df.columns:
        print(f"{col:<30} | {str(df[col].dtype):<10} | {df[col].isnull().sum()}")
    print("="*60 + "\n")

# Execute Phase 1
df = ingest_and_structure_data(DATA_FILE)
if df is not None:
    generate_data_audit_report(df)
    print(">> PHASE 1 COMPLETED. Data is ready for modeling.")
else:
    sys.exit("!! SYSTEM EXIT: Data loading failed.")

## Phase 2: Comprehensive Multi-Model Estimation Engine

Estimates 9 models (6 SARIMAX + 1 Logit + 2 ancillary SARIMAX) to test the merit-order and volatility hypotheses. Results are cached to the `Phase 2` directory to avoid redundant computation on re-runs.

In [ ]:
# =============================================================================
# PHASE 2: COMPREHENSIVE MULTI-MODEL ESTIMATION ENGINE (CACHED)
# =============================================================================
import pandas as pd
import statsmodels.api as sm
import time
import os

RESULTS_DIR_P2 = os.path.join(RESULTS_BASE, "Phase_2")
os.makedirs(RESULTS_DIR_P2, exist_ok=True)

def generate_interaction_terms(df: pd.DataFrame) -> tuple:
    data = df.copy()
    if 'Sunday' not in data.columns:
        data['Sunday'] = (data.index.dayofweek == 6).astype(int)

    data['Interaction_Solar_Sunday'] = data['Solar_Generation_Share'] * data['Sunday']
    data['Interaction_Solar_Holiday'] = data['Solar_Generation_Share'] * data.get('Public_Holiday', 0)

    hourly_cols = []
    for h in range(8, 18):
        col_name = f'Solar_Share_x_Hour_{h}'
        data[col_name] = data['Solar_Generation_Share'] * (data['Hour'] == h).astype(int)
        hourly_cols.append(col_name)

    data['Interaction_Solar_Peak'] = data['Solar_Generation_Share'] * ((data['Hour'] >= 10) & (data['Hour'] <= 15)).astype(int)
    data['Interaction_Solar_Dir_Down'] = data['Solar_Generation_Share'] * data['System_Direction_Down']
    data['Interaction_Solar_Dir_Up'] = data['Solar_Generation_Share'] * data.get('System_Direction_Up', 0)

    return data, hourly_cols

def run_full_analysis_suite(df: pd.DataFrame):
    df_processed, hourly_cols = generate_interaction_terms(df)
    
    base_controls = [
        'Consumption', 'Wind_Generation', 'Import_Coal_Generation',
        'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
        'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday'
    ]

    models_config = [
        {'id': 'Model 1', 'name': 'a-FRR Baseline', 'type': 'SARIMAX', 'y': 'a-FRR_Price', 'x_vars': base_controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']},
        {'id': 'Model 4', 'name': 'MCP Baseline', 'type': 'SARIMAX', 'y': 'Mcp', 'x_vars': base_controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']},
        {'id': 'Model 7.1', 'name': 'Surplus Prob (Logit)', 'type': 'LOGIT', 'y': 'System_Direction_Down', 'x_vars': ['Solar_Generation_Share'] + base_controls}
    ]

    print(f"\n{'='*80}\nSTARTING PHASE 2 ANALYSIS: {len(models_config)} MODELS QUEUED (DIR: {RESULTS_DIR_P2})\n{'='*80}")

    for idx, config in enumerate(models_config, 1):
        model_id = config['id']
        safe_id = model_id.replace(" ", "_")
        csv_path = os.path.join(RESULTS_DIR_P2, f"{safe_id}_summary.csv")
        
        print(f">> [{idx}/{len(models_config)}] Initiating {model_id}: {config['name']}...")
        
        # NaN değerleri atarak veri setini modellemeye hazır hale getir
        model_data = df_processed[[config['y']] + config['x_vars']].dropna()
        endog, exog = model_data[config['y']], model_data[config['x_vars']]

        # Cache check and display results
        if os.path.exists(csv_path):
            mode_str = "Logistic Regression" if config['type'] == 'LOGIT' else "SARIMAX(1,1,1)(1,0,1)[24]"
            print(f"   Mode: {mode_str} | Obs: {len(endog)}")
            print(f"   [CACHE] Results loaded from '{RESULTS_DIR_P2}'.")
            
            try:
                cached_summary = pd.read_csv(csv_path)
                print("-" * 60 + f"\nRESULTS SUMMARY FOR {model_id}\n" + "-" * 60)
                print(cached_summary.to_string(index=False))
                print("=" * 80 + "\n")
            except Exception as e:
                print(f"   !! Cache okuma hatası: {e}\n" + "=" * 80 + "\n")
            continue

        # Cache yoksa modeli yeniden tahmin et
        try:
            t_start = time.time()
            if config['type'] == 'LOGIT':
                print(f"   Mode: Logistic Regression | Obs: {len(endog)}")
                model = sm.Logit(endog, sm.add_constant(exog, prepend=True))
            else:
                print(f"   Mode: SARIMAX(1,1,1)(1,0,1)[24] | Obs: {len(endog)}")
                model = sm.tsa.SARIMAX(endog, exog=exog, order=(1, 1, 1), seasonal_order=(1, 0, 1, 24), enforce_stationarity=False, enforce_invertibility=False)
            
            res = model.fit(disp=False)
            print(f"   Converged in {time.time() - t_start:.2f} seconds.")
            
            print("-" * 60 + f"\nRESULTS SUMMARY FOR {model_id}\n" + "-" * 60)
            summary_table = res.summary().tables[1]
            print(summary_table)
            
            # Sonuçları CSV'ye kaydet
            summary_df = pd.DataFrame(summary_table.data[1:], columns=summary_table.data[0])
            summary_df.to_csv(csv_path, index=False)
            print("=" * 80 + "\n")
            
        except Exception as e:
            print(f"   !!! ERROR in {config['id']}: {e}\n" + "=" * 80 + "\n")

# Analizi tetikle
run_full_analysis_suite(df)

## Phase 3: Parameter Justification and Optimization
Performs ACF/PACF diagnostics to confirm the `s=24` hourly seasonality, then runs a grid search over SARIMAX parameter space to identify optimal `(p,d,q)(P,D,Q)s` specifications. Results are cached and printed if already computed.

In [ ]:
# =============================================================================
# PHASE 3: PARAMETER JUSTIFICATION AND OPTIMIZATION (CACHED)
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools
import warnings
import sys
import time
import os

warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_3")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

CSV_PATH = os.path.join(OUTPUT_DIR, "Phase3_Optimal_Parameters.csv")
DIAG_PLOT_PATH = os.path.join(OUTPUT_DIR, "Phase3_Seasonality_Diagnostics.png")

# =============================================================================
# PART 1: SEASONALITY DIAGNOSTICS (ACF/PACF)
# =============================================================================

def perform_seasonality_diagnostics(df):
    """
    Generates ACF/PACF plots for seasonality validation.
    Titles are omitted from saved figures to meet academic standards.
    """
    print("\n" + "="*80)
    print("PHASE 3.1: SEASONALITY DIAGNOSTICS (ACF/PACF ANALYSIS)")
    print("="*80)

    try:
        # Representative weekly samples across seasons
        winter = df[(df.index >= '2022-01-10') & (df.index < '2022-01-17')]
        autumn = df[(df.index >= '2022-10-10') & (df.index < '2022-10-17')]
        spring = df[(df.index >= '2023-04-10') & (df.index < '2023-04-17')]
        summer = df[(df.index >= '2024-07-08') & (df.index < '2024-07-15')]

        subset = pd.concat([winter, autumn, spring, summer])
        print(f">> Representative subset created. Observations: {len(subset)}")
    except Exception as e:
        print(f"!! Sampling Error: {e}")
        return

    target_vars = ['Solar_Generation_Share', 'a-FRR_Price', 'Mcp', 'Consumption']
    fig, axes = plt.subplots(len(target_vars), 2, figsize=(16, 16))
    plt.subplots_adjust(hspace=0.4)

    print(f"{'VARIABLE':<30} | {'LAG-24 AUTOCORRELATION':<20}")
    print("-" * 60)

    for i, var in enumerate(target_vars):
        if var not in subset.columns: continue
        series = subset[var].dropna()

        plot_acf(series, lags=72, ax=axes[i, 0], title='')
        axes[i, 0].axvline(x=24, color='red', linestyle='--', alpha=0.8, label='$s=24$ Seasonality')
        axes[i, 0].legend(loc='upper right')

        plot_pacf(series, lags=72, ax=axes[i, 1], title='')
        axes[i, 1].axvline(x=24, color='red', linestyle='--', alpha=0.8, label='$s=24$ Seasonality')
        axes[i, 1].legend(loc='upper right')

        acf_vals = sm.tsa.acf(series, nlags=72)
        print(f"{var:<30} | {acf_vals[24]:.4f}")

    plt.savefig(DIAG_PLOT_PATH, dpi=FIG_DPI, bbox_inches='tight')
    print(f">> Diagnostic plots saved to '{DIAG_PLOT_PATH}'")
    plt.show()

# =============================================================================
# PART 2: GRID SEARCH OPTIMIZATION
# =============================================================================

def execute_grid_search_optimization(df):
    """
    Runs the SARIMAX grid search algorithm over the candidate parameter space.
    Results are saved to CSV_PATH upon completion.
    """
    print("\n" + "="*80)
    print("PHASE 3.2: HYPERPARAMETER OPTIMIZATION (GRID SEARCH RUNNING)")
    print("="*80)

# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))

    if target_df is not None:
        global_start = time.time()

        perform_seasonality_diagnostics(target_df)

        if os.path.exists(CSV_PATH):
            print("\n" + "="*80)
            print(f"PHASE 3.2: HYPERPARAMETER OPTIMIZATION (LOADED FROM '{OUTPUT_DIR}')")
            print("="*80)
            print(f">> Cached results found. Skipping computation...")

            results_df = pd.read_csv(CSV_PATH, index_col=0)

            print("\n" + "#" * 60)
            print("PHASE 3 RESULTS: OPTIMAL MODEL SPECIFICATIONS")
            print("#" * 60)

            for name, row in results_df.iterrows():
                report_str = (f"\nModel: {name}\n"
                              f"   Best Order (p,d,q)   : {row['Order']}\n"
                              f"   Best Seasonal (P,D,Q): {row['Seasonal']} (s=24)\n"
                              f"   AIC Score            : {row['AIC']:.4f}\n"
                              f"{'-' * 60}")
                print(report_str)
        else:
            execute_grid_search_optimization(target_df)

        duration = (time.time() - global_start) / 60
        print("\n" + "="*80)
        print(f"PHASE 3 COMPLETED IN {duration:.2f} MINUTES.")
        print("="*80)
    else:
        print("!! Critical Error: DataFrame 'df' not found. Please execute Phase 1.")

## Phase 4: Robust Estimation with Optimized Parameters

Re-estimates the core SARIMAX models using the optimal parameters identified in Phase 3. Saves full regression summaries, coefficient tables, and diagnostic metrics.

In [ ]:
# =============================================================================
# PHASE 4: ROBUST ESTIMATION WITH OPTIMIZED PARAMETERS (CACHED)
# =============================================================================
import pandas as pd
import statsmodels.api as sm
import os

OUTPUT_DIR_P4 = os.path.join(RESULTS_BASE, "Phase_4")
os.makedirs(OUTPUT_DIR_P4, exist_ok=True)

REPORT_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_Full_Estimation_Report.txt")
COEF_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_All_Coefficients.csv")
DIAG_FILE_P4 = os.path.join(OUTPUT_DIR_P4, "Phase4_Model_Diagnostics.csv")

def run_phase_4(df):
    print("\n" + "="*80 + "\nPHASE 4: ROBUST ESTIMATION PROCESS STARTED\n" + "="*80)


    if os.path.exists(REPORT_FILE_P4) and os.path.exists(COEF_FILE_P4):
        print(f">> [CACHE] Results found in '{OUTPUT_DIR_P4}'. Skipping estimation.\n>> Loading cached report...\n" + "-" * 80)
        with open(REPORT_FILE_P4, "r", encoding="utf-8") as f:
            print(f.read())
        print("-" * 80 + f"\n>> Coefficients loaded from '{COEF_FILE_P4}'.")
        return

    print(">> No cached results found. Running SARIMAX estimation (est. 5-15 min)...")

    ORDER_MCP, SEASONAL_MCP = (2, 0, 2), (1, 0, 1, 24)
    ORDER_aFRR, SEASONAL_aFRR = (1, 1, 2), (1, 0, 1, 24)

    # Özellik mühendisliği (Eksikse tamamla)
    if 'Interaction_Solar_Sunday' not in df.columns:
        df['Sunday'] = (df.index.dayofweek == 6).astype(int)
        df['Interaction_Solar_Sunday'] = df['Solar_Generation_Share'] * df['Sunday']
        df['Interaction_Solar_Holiday'] = df['Solar_Generation_Share'] * df.get('Public_Holiday', 0)
        for h in range(8, 18):
            df[f'Solar_Share_x_Hour_{h}'] = df['Solar_Generation_Share'] * (df['Hour'] == h).astype(int)
        df['Interaction_Solar_Peak'] = df['Solar_Generation_Share'] * ((df['Hour'] >= 10) & (df['Hour'] <= 15)).astype(int)
        df['Interaction_Solar_Dir_Down'] = df['Solar_Generation_Share'] * df['System_Direction_Down']
        df['Interaction_Solar_Dir_Up'] = df['Solar_Generation_Share'] * df.get('System_Direction_Up', 0)

    common_controls = [
        'Consumption', 'Wind_Generation', 'Import_Coal_Generation', 
        'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap', 
        'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos', 
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday'
    ]
    hourly_cols = [c for c in df.columns if 'Solar_Share_x_Hour_' in c]

    models_config = [
        {'id': 'Model 1', 'name': 'a-FRR Baseline', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']},
        {'id': 'Model 2', 'name': 'a-FRR Hourly', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + hourly_cols},
        {'id': 'Model 7.3', 'name': 'a-FRR Flexibility Trap', 'y': 'a-FRR_Price', 'params': 'aFRR', 'X': common_controls + ['Solar_Generation_Share', 'System_Direction_Down', 'System_Direction_Up', 'Interaction_Solar_Dir_Down', 'Interaction_Solar_Dir_Up']},
        {'id': 'Model 4', 'name': 'MCP Baseline', 'y': 'Mcp', 'params': 'MCP', 'X': common_controls + ['Solar_Generation_Share', 'Interaction_Solar_Sunday', 'Interaction_Solar_Holiday']},
        {'id': 'Model 5', 'name': 'MCP Hourly', 'y': 'Mcp', 'params': 'MCP', 'X': common_controls + hourly_cols}
    ]

    coefficient_frames, diagnostic_summary = [], []

    with open(REPORT_FILE_P4, "w", encoding="utf-8") as f:
        f.write("PHASE 4: FULL ESTIMATION RESULTS (OPTIMIZED PARAMETERS)\n" + "="*80 + "\n\n")
        
        for idx, config in enumerate(models_config, 1):
            print(f">> [{idx}/{len(models_config)}] Running {config['id']}: {config['name']}...")
            order, seasonal = (ORDER_MCP, SEASONAL_MCP) if config['params'] == 'MCP' else (ORDER_aFRR, SEASONAL_aFRR)

            # Eksik verileri filtrele
            data_mod = df[[config['y']] + config['X']].dropna()
            
            try:
                model = sm.tsa.SARIMAX(
                    endog=data_mod[config['y']], 
                    exog=data_mod[config['X']], 
                    order=order, 
                    seasonal_order=seasonal, 
                    enforce_stationarity=False, 
                    enforce_invertibility=False
                )
                res = model.fit(disp=False)

                f.write(f"\n{'#'*90}\nRESULTS FOR {config['id']}: {config['name']}\n{'#'*90}\n{res.summary()}\n")
                print("-" * 90 + f"\nRESULTS SUMMARY FOR {config['id']}\n" + "-" * 90)
                print(res.summary().tables[1])

                coef_df = res.params.to_frame(name='Coefficient')
                coef_df['Model_ID'] = config['id']
                coefficient_frames.append(coef_df)
                diagnostic_summary.append({'Model_ID': config['id'], 'AIC': res.aic, 'BIC': res.bic})
            except Exception as e:
                print(f"   !!! ERROR in {config['id']}: {e}")

    if coefficient_frames:
        pd.concat(coefficient_frames).to_csv(COEF_FILE_P4)
        pd.DataFrame(diagnostic_summary).to_csv(DIAG_FILE_P4, index=False)
    print(f"\n{'='*80}\nPHASE 4 COMPLETED. Outputs saved to '{OUTPUT_DIR_P4}'.")

# Çalıştırma
if 'df' in globals() and df is not None:
    run_phase_4(df)

## Phase 5: Advanced Visualization (SARIMAX Results)


In [ ]:
# =============================================================================
# PHASE 5: ADVANCED VISUALIZATION (SARIMAX) - FULLY DYNAMIC EXPORT MODE
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")

# Output directory configuration
if os.path.isdir("results"):
    RESULTS_BASE = "results"
elif os.path.isdir(os.path.join("..", "results")):
    RESULTS_BASE = os.path.join("..", "results")
else:
    RESULTS_BASE = "results"

OUTPUT_DIR_P5 = os.path.join(RESULTS_BASE, "Phase_5")
COEF_FILE_P4 = os.path.join(RESULTS_BASE, "Phase_4", "Phase4_All_Coefficients.csv")
os.makedirs(OUTPUT_DIR_P5, exist_ok=True)

# Academic visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def get_dynamic_coefficients(csv_path: str, model_id: str) -> dict:
    """
    Reads hourly Solar_Share_x_Hour coefficients from the Phase 4 CSV output 
    for a given model and returns them as a dictionary.
    
    Args:
        csv_path (str): Path to the Phase 4 coefficient file.
        model_id (str): Model ID to extract (e.g. 'Model 2', 'Model 5').
        
    Returns:
        dict: {hour (int): coefficient (float)} dictionary.
    """
    coeffs = {}
    try:
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Coefficient file not found: {csv_path}")
            
        df_coef = pd.read_csv(csv_path, index_col=0)
        
        # Filter coefficients for the target model
        model_data = df_coef[df_coef['Model_ID'] == model_id]
        
        if model_data.empty:
            raise ValueError(f"Model '{model_id}' not found in coefficient file.")
            
        # Extract hourly coefficients from hour 8 to 17
        for h in range(8, 18):
            col_name = f'Solar_Share_x_Hour_{h}'
            if col_name in model_data.index:
                val = model_data.loc[col_name, 'Coefficient']
                # Take first value if multiple matches (duplicate index guard)
                if isinstance(val, pd.Series):
                    val = val.iloc[0]
                coeffs[h] = float(val)
                
    except Exception as e:
        print(f"!! COEFFICIENT READ ERROR ({model_id}): {e}")
        
    return coeffs

def run_phase_5_final(df: pd.DataFrame):
    """
    Visualizes market profiles and SARIMAX coefficients.
    Dynamically reads coefficients from Phase 4 outputs instead of hardcoding.
    """
    print(f">> Phase 5: Generating dynamic publication-ready figures to '{OUTPUT_DIR_P5}'...")

    # Datetime index safety check
    if 'Date' in df.columns and not np.issubdtype(df['Date'].dtype, np.datetime64):
        df['Date'] = pd.to_datetime(df['Date'])
    elif isinstance(df.index, pd.DatetimeIndex):
        df['Date'] = df.index

    # =========================================================================
    # FIGURE 1 & 2: DAILY MARKET PROFILE
    # =========================================================================
    h_sum = df.groupby('Hour').agg(
        M_Sol=('Solar_Generation', 'mean'),
        Q25=('Solar_Generation', lambda x: x.quantile(0.25)),
        Q75=('Solar_Generation', lambda x: x.quantile(0.75)),
        M_MCP=('Mcp', 'mean'),
        M_aFRR=('a-FRR_Price', 'mean')
    ).reset_index()

    def plot_profile(data, title, filename):
        fig, ax1 = plt.subplots(figsize=(12, 6))

        ax1.set_xlabel('Hour of the Day', fontweight='bold')
        ax1.set_ylabel('Average Generation (MWh)', color='teal', fontweight='bold')
        ax1.plot(data['Hour'], data['M_Sol'], color='teal', linewidth=2.5, label='Solar Generation')
        ax1.fill_between(data['Hour'], data['Q25'], data['Q75'], color='teal', alpha=0.15)
        ax1.tick_params(axis='y', labelcolor='teal')

        ax2 = ax1.twinx()
        ax2.set_ylabel('Average MCP (TL/MWh)', color='darkorange', fontweight='bold')
        ax2.plot(data['Hour'], data['M_MCP'], color='darkorange', linewidth=2.5, label='MCP')
        ax2.tick_params(axis='y', labelcolor='darkorange')

        # [Revision_Review_Referee_2.7 / MÖK-4] the a-FRR capacity price (TL/MW) is not commensurable with the
        # MCP (TL/MWh): it gets its own right-hand axis instead of sharing the MCP axis.
        ax3 = ax1.twinx()
        ax3.spines['right'].set_position(('axes', 1.09))
        ax3.spines['right'].set_visible(True)
        ax3.set_ylabel('Average a-FRR Capacity Price (TL/MW)', color='crimson', fontweight='bold')
        ax3.plot(data['Hour'], data['M_aFRR'], color='crimson', linestyle='--', linewidth=2.5, label='a-FRR Price')
        ax3.tick_params(axis='y', labelcolor='crimson')
        ax3.grid(False)

        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        lines3, labels3 = ax3.get_legend_handles_labels()
        ax3.legend(lines + lines2 + lines3, labels + labels2 + labels3, loc='upper center', bbox_to_anchor=(.5, 1.08), ncol=3, frameon=False)

        plt.xticks(np.arange(0, 24, 1))
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR_P5, filename), dpi=FIG_DPI, bbox_inches='tight')
        plt.show()

    plot_profile(h_sum, "Full Dataset", 'Phase_5_Figure_1_Daily_Market_Profile.png')

    if 'Date' in df.columns:
        print(">> Generating 2025 subset profile...")
        df_2025 = df[df['Date'].dt.year == 2025].copy()
        if not df_2025.empty:
            h_sum_25 = df_2025.groupby('Hour').agg(
                M_Sol=('Solar_Generation', 'mean'),
                Q25=('Solar_Generation', lambda x: x.quantile(0.25)),
                Q75=('Solar_Generation', lambda x: x.quantile(0.75)),
                M_MCP=('Mcp', 'mean'),
                M_aFRR=('a-FRR_Price', 'mean')
            ).reset_index()
            plot_profile(h_sum_25, "2025", 'Phase_5_Figure_2_Daily_Market_Profile.png')

    # =========================================================================
    # FIGURE 4 & 6: DYNAMIC SARIMAX HOURLY IMPACT COEFFICIENTS
    # =========================================================================
    print(f">> Extracting dynamic coefficients from {COEF_FILE_P4}...")
    
    # Dynamic coefficient extraction for Model 5 (MCP) and Model 2 (a-FRR)
    ptf_coeffs = get_dynamic_coefficients(COEF_FILE_P4, 'Model 5')
    sfk_coeffs = get_dynamic_coefficients(COEF_FILE_P4, 'Model 2')

    configs = []
    if ptf_coeffs:
        configs.append((ptf_coeffs, 'Phase_5_Figure_4_Hourly_Merit_Order_Effect_MCP.png', 'mako', 'SARIMAX Impact on MCP (TL/MWh)'))
    if sfk_coeffs:
        configs.append((sfk_coeffs, 'Phase_5_Figure_6_Hourly_Volatility_Effect_aFRR.png', 'rocket_r', 'SARIMAX Impact on a-FRR (TL/MW)'))  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)

    if not configs:
        print("!! WARNING: Could not read dynamic coefficients. Bar plots skipped.")
    else:
        for coeffs, filename, pal, y_label in configs:
            plt.figure(figsize=(10, 5))
            ser = pd.Series(coeffs)
            sns.barplot(x=ser.index, y=ser.values, palette=pal)
            
            plt.axhline(0, color='black', linewidth=1.2)
            plt.xlabel('Hour of the Day', fontweight='bold')
            plt.ylabel(y_label, fontweight='bold')
            plt.tight_layout()

            plt.savefig(os.path.join(OUTPUT_DIR_P5, filename), dpi=FIG_DPI, bbox_inches='tight')
            plt.show()

    print(f"\n>> Phase 5 completed dynamically. Outputs saved to '{OUTPUT_DIR_P5}'.")

# --- Execution ---
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_phase_5_final(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found in memory. Please execute Phase 1 first.")

### Phase 6.1: Partial Dependence Analysis


In [ ]:
# =================================================================================================
# PHASE 6.1: NON-LINEAR IMPACT ANALYSIS (XGBOOST) - REVISED (DECILE LINES REMOVED)
# =================================================================================================
# [Revision_Review_Referee_2.7] Axis labels: PartialDependenceDisplay.from_estimator(ax=...) replaces the
# Axes it is given, so labels set on ax5/ax7/ax1 never reached the saved figures (sklearn defaults
# 'Partial dependence' / 'Solar_Generation_Share' were shown). Labels are now set on disp.axes_[0, 0]
# and carry the price units (MCP: TL/MWh; a-FRR capacity price: TL/MW). The PDP grid values are also
# exported (Phase_6_1_PDP_Values.csv) so the curves can be compared numerically across re-runs.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.inspection import PartialDependenceDisplay
import os
import warnings

warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_6")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def check_phase6_plots():
    """Checks if the required individual plot files already exist."""
    fig_5 = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png')
    fig_7 = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png')
    fig_8 = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')
    pdp_csv = os.path.join(OUTPUT_DIR, 'Phase_6_1_PDP_Values.csv')   # [Revision] the numerical record is part of the cache
    return os.path.exists(fig_5) and os.path.exists(fig_7) and os.path.exists(fig_8) and os.path.exists(pdp_csv)

def run_phase_6_1_final(df):
    print(f"PHASE 6.1: ML Analysis (Directory: '{OUTPUT_DIR}')...")
    print("="*80)

    if check_phase6_plots():
        print(f">> [CACHE] Existing analyses found in '{OUTPUT_DIR}'.")
        print(">> Skipping XGBoost training, loading saved plots...\n")

        fig_5_path = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png')
        fig_7_path = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png')
        fig_8_path = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')

        plt.figure(figsize=(10, 6)); plt.imshow(plt.imread(fig_5_path)); plt.axis('off')
        plt.title("Phase 6.1 - Figure 5 (Preview)", fontsize=14); plt.show()

        plt.figure(figsize=(10, 6)); plt.imshow(plt.imread(fig_7_path)); plt.axis('off')
        plt.title("Phase 6.1 - Figure 7 (Preview)", fontsize=14); plt.show()

        plt.figure(figsize=(16, 6)); plt.imshow(plt.imread(fig_8_path)); plt.axis('off')
        plt.title("Phase 6.1 - Figure 8 (Preview)", fontsize=14); plt.show()

        print(f">> Cached data from '{OUTPUT_DIR}' loaded successfully.")
        print(">> To regenerate, delete the PNG files and re-run this cell.")
        return

    print(">> No cached results found. Training XGBoost models...")

    df_ml = df.copy()
    if df_ml['Solar_Generation_Share'].max() <= 1.0:
        df_ml['Solar_Generation_Share'] *= 100

    features_core = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Price_Cap', 'Hour_Sin', 'Hour_Cos']
    features_with_mcp = features_core + ['Mcp']

    m_a = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['Mcp'])
    m_b = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_with_mcp], df_ml['a-FRR_Price'])
    m_c = xgb.XGBClassifier(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['System_Direction_Down'])

    # FIGURE 5: MCP MERIT ORDER EFFECT
    print(">> Plotting Figure 5 (MCP)...")
    fig5, ax5 = plt.subplots(figsize=(10, 6))
    disp5 = PartialDependenceDisplay.from_estimator(
        m_a, df_ml[features_core], ['Solar_Generation_Share'],
        ax=ax5, line_kw={"color": "orange", "linewidth": 3}
    )
    # Remove decile markers for cleaner output
    for vlines in disp5.deciles_vlines_:
        for v in vlines:
            v.set_visible(False)
    _ax5 = disp5.axes_[0, 0]   # [Revision_Review_Referee_2.7] the Axes actually drawn on
    _ax5.set_xlabel('Solar Generation Share (%)')
    _ax5.set_ylabel('Partial dependence of MCP (TL/MWh)')
    plt.tight_layout()
    save_path_fig5 = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_5_PDP_Merit_Order_and_Volatility.png')
    plt.savefig(save_path_fig5, dpi=FIG_DPI, bbox_inches='tight')
    plt.title("Phase 6.1 - Figure 5: MCP Merit-Order Effect (Preview Only)", fontsize=14)
    plt.show()

    # FIGURE 7: a-FRR VOLATILITY EFFECT
    print(">> Plotting Figure 7 (a-FRR)...")
    fig7, ax7 = plt.subplots(figsize=(10, 6))
    disp7 = PartialDependenceDisplay.from_estimator(
        m_b, df_ml[features_with_mcp], ['Solar_Generation_Share'],
        ax=ax7, line_kw={"color": "crimson", "linewidth": 3}
    )
    for vlines in disp7.deciles_vlines_:
        for v in vlines:
            v.set_visible(False)
    _ax7 = disp7.axes_[0, 0]   # [Revision_Review_Referee_2.7]
    _ax7.set_xlabel('Solar Generation Share (%)')
    _ax7.set_ylabel('Partial dependence of a-FRR price (TL/MW)')
    plt.tight_layout()
    save_path_fig7 = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_7_PDP_Merit_Order_and_Volatility.png')
    plt.savefig(save_path_fig7, dpi=FIG_DPI, bbox_inches='tight')
    plt.title("Phase 6.1 - Figure 7: a-FRR Volatility Effect (Preview Only)", fontsize=14)
    plt.show()

    # FIGURE 8: SURPLUS ANALYSIS
    print(">> Plotting Figure 8 (Surplus Analysis)...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    disp8 = PartialDependenceDisplay.from_estimator(
        m_c, df_ml[features_core], ['Solar_Generation_Share'],
        ax=ax1, line_kw={"color": "blue", "linewidth": 3}
    )
    for vlines in disp8.deciles_vlines_:
        for v in vlines:
            v.set_visible(False)
    _ax1 = disp8.axes_[0, 0]   # [Revision_Review_Referee_2.7]
    _ax1.set_xlabel('Solar Generation Share (%)')
    _ax1.set_ylabel('Predicted probability of energy surplus')

    sns.boxplot(x=df_ml['System_Direction_Down'], y=df_ml['Solar_Generation_Share'], palette="coolwarm", ax=ax2)
    ax2.set_xticklabels(['Balanced/Deficit', 'Surplus'])
    ax2.set_xlabel('System State')
    ax2.set_ylabel('Solar Generation Share (%)')
    plt.tight_layout()
    save_path_surplus = os.path.join(OUTPUT_DIR, 'Phase_6_Figure_8_Surplus_Probability_and_Distribution.png')
    plt.savefig(save_path_surplus, dpi=FIG_DPI, bbox_inches='tight')
    plt.suptitle("Phase 6.1 - Figure 8: Surplus Analysis (Preview Only)", fontsize=16)
    plt.show()

    # [Revision_Review_Referee_2.7] export the PDP grid values (numerical record of Figures 4, 6, 7-left)
    def _pd_xy(disp):
        r = disp.pd_results[0]
        grid = r['grid_values'][0] if 'grid_values' in r else r['values'][0]
        return np.asarray(grid), np.asarray(r['average'][0])
    _rows = []
    for _name, _disp in [('MCP (Figure 4)', disp5), ('a-FRR price (Figure 6)', disp7), ('P(surplus) (Figure 7, left)', disp8)]:
        _x, _y = _pd_xy(_disp)
        _rows += [{'Figure': _name, 'Solar_Generation_Share_pct': float(a), 'Partial_dependence': float(b)} for a, b in zip(_x, _y)]
    _pdp_path = os.path.join(OUTPUT_DIR, 'Phase_6_1_PDP_Values.csv')
    pd.DataFrame(_rows).to_csv(_pdp_path, index=False, **CSV_KW)
    print(f">> PDP grid values saved to '{_pdp_path}'.")

    print(f"\n>> All ML analyses and figures saved to '{OUTPUT_DIR}'.")

if __name__ == "__main__":
    if 'df' in locals():
        run_phase_6_1_final(df)
    elif 'df' in globals():
        run_phase_6_1_final(globals()['df'])

### Phase 6.2: Flexibility Trap Analysis


In [ ]:
# =============================================================================
# PHASE 6.2: NON-LINEAR IMPACT ANALYSIS (XGBOOST) - FLEXIBILITY TRAP
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import os
import warnings

warnings.filterwarnings("ignore")

OUTPUT_DIR_P6 = os.path.join(RESULTS_BASE, "Phase_6")
if not os.path.exists(OUTPUT_DIR_P6):
    os.makedirs(OUTPUT_DIR_P6)

# Academic plot settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def check_phase6_2_plot():
    """Checks if Figure 9 (Flexibility Trap) already exists."""
    fig_9 = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_9_Flexibility_Trap.png')
    return os.path.exists(fig_9)

def run_phase_6_2_final(df):
    print(f">> Phase 6.2: Flexibility Trap Analysis (Directory: '{OUTPUT_DIR_P6}')...")
    print("="*80)

    fig_9_path = os.path.join(OUTPUT_DIR_P6, 'Phase_6_Figure_9_Flexibility_Trap.png')

    # Cache check
    if check_phase6_2_plot():
        print(f">> [CACHE] Existing analysis found in '{OUTPUT_DIR_P6}'.")
        print(">> Skipping XGBoost training, loading saved plot...\n")

        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(fig_9_path))
        plt.axis('off')
        plt.title("Phase 6.2 - Figure 9: Flexibility Trap (Preview)", fontsize=14)
        plt.show()

        print(f">> Cached data from '{OUTPUT_DIR_P6}' loaded successfully.")
        print(">> To regenerate, delete the PNG file and re-run this cell.")
        return

    print(">> No cached results found. Training XGBoost models (Flexibility Trap)...")

    # Data preparation
    dm = df.copy()
    if dm['Solar_Generation_Share'].max() <= 1:
        dm['Solar_Generation_Share'] *= 100

    sur = dm[dm['System_Direction_Down'] == 1]
    dfc = dm[dm['System_Direction_Down'] == 0]

    ft = ['Solar_Generation_Share', 'Mcp', 'Consumption', 'Wind_Generation', 'Hour_Sin', 'Hour_Cos']

    # Model training
    print(">> Training conditional XGBoost models...")
    ms = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(sur[ft], sur['a-FRR_Price'])
    md = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(dfc[ft], dfc['a-FRR_Price'])

    # Ceteris paribus simulation
    sr = np.linspace(0, 60, 100)
    avg = dm[ft].median()
    sim = pd.DataFrame([avg]*100, columns=ft)
    sim['Solar_Generation_Share'] = sr

    mk = sr <= 45
    yd = md.predict(sim[mk][ft])
    ys = ms.predict(sim[ft])
    ysc = ys[:len(yd)]
    sc = sr[mk]

    # -------------------------------------------------------------------------
    # FIGURE 9: FLEXIBILITY TRAP
    # -------------------------------------------------------------------------
    print("  [Fig 9] Plotting Flexibility Trap...")
    fig, ax = plt.subplots(figsize=(10, 6))


    ax.plot(sc, yd, color='green', lw=3.0, label='Energy Deficit & Balanced')
    ax.plot(sr, ys, color='red', lw=3.0, label='Energy Surplus')
    
    # Zone 1: Elasticity Risk Gap
    ax.fill_between(
        sc, ysc, yd, 
        where=(yd >= ysc), interpolate=True, 
        color='gray', alpha=.15, hatch='...', 
        label='Elasticity Risk Gap'
    )
    
    # Zone 2: Flexibility Trap Zone
    ax.fill_between(
        sc, ysc, yd, 
        where=(ysc > yd), interpolate=True, 
        color='red', alpha=.10, hatch='xxx', 
        label='Flexibility Trap Zone'
    )

    ax.set_xlabel('Solar Generation Share (%)', fontweight='bold')
    ax.set_ylabel('Predicted a-FRR Price (TL/MW)', fontweight='bold')  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)
    ax.legend(loc='upper left', fontsize=11, frameon=True)
    ax.axvline(x=45, color='green', ls=':', alpha=.5)

    plt.tight_layout()
    plt.savefig(fig_9_path, dpi=FIG_DPI, bbox_inches='tight')
    plt.show()

    # Zone dominance statistics
    trap_pct = (ysc > yd).sum() / len(yd) * 100
    gap_pct = (yd >= ysc).sum() / len(yd) * 100
    print(f"\n>> Zone dominance in the 0-45% solar range:")
    print(f"   - Elasticity Risk Gap (Deficit > Surplus): {gap_pct:.1f}%")
    print(f"   - Flexibility Trap Zone (Surplus > Deficit): {trap_pct:.1f}%")

    print(f"\n>> Phase 6.2 complete. Flexibility Trap figure saved to '{OUTPUT_DIR_P6}'.")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    if 'df' in globals() and df is not None:
        run_phase_6_2_final(df)
    else:
        print("ERROR: DataFrame 'df' not found. Please run Phase 1 first.")

### Phase 6.4: Out-of-Sample Validation of the XGBoost Cross-Checks (Revision_Review_Referee_2.3)

XGBoost is used as an interpretive cross-check (partial-dependence plots), not as a forecasting model. To show that
the patterns are not in-sample artefacts, held-out performance is evaluated with 5-fold expanding-window
(walk-forward) time-series cross-validation (`sklearn.model_selection.TimeSeriesSplit`): always train on the past,
test on the future. The models evaluated are **exactly** the Phase 6.1 models — same features (`features_core`,
`features_with_mcp`) and the same estimator call (`n_estimators=200, random_state=42`, XGBoost library defaults
otherwise) — so the metrics refer to the models whose partial-dependence plots are interpreted (Figures 4, 6, 7).
Regression targets report MAE, RMSE and out-of-sample R²; the system-direction classifier reports accuracy and F1.
The in-sample fit of the full-sample model is reported alongside for comparison.

Produces: `Phase_6_4_Out_of_Sample_CV.csv` (fold averages) and `Phase_6_4_Out_of_Sample_CV_Folds.csv` (per fold) in `results/Phase_6/`.


In [ ]:
# =============================================================================
# PHASE 6.4: OUT-OF-SAMPLE VALIDATION (EXPANDING-WINDOW TIME-SERIES CV) - CACHED
# =============================================================================
# Revision_Review_Referee_2.3 — Out-of-sample validation of XGBoost
# Added in revision. Feeds Supplementary Appendix J.3.
# Follow-up: evaluates EXACTLY the Phase 6.1 models (same features, same estimator
# call: n_estimators=200, random_state=42, XGBoost library defaults otherwise), so
# that the held-out metrics refer to the models whose partial-dependence plots are
# interpreted (Figures 4, 6, 7). Per-fold metrics and the in-sample fit of the
# full-sample model are reported alongside.
# =============================================================================
import pandas as pd, numpy as np, os, warnings
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (mean_absolute_error, mean_squared_error, accuracy_score, f1_score,
                             roc_auc_score, balanced_accuracy_score)
warnings.filterwarnings("ignore")

OUTPUT_DIR_P6 = os.path.join(RESULTS_BASE, "Phase_6")
os.makedirs(OUTPUT_DIR_P6, exist_ok=True)

# Identical to Phase 6.1 (features_core / features_with_mcp and the estimator call)
CV_FEATURES_CORE = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Price_Cap', 'Hour_Sin', 'Hour_Cos']
CV_FEATURES_AFRR = CV_FEATURES_CORE + ['Mcp']
CV_XGB_PARAMS = dict(n_estimators=200, random_state=42)   # as in Phase 6.1; every other XGBoost setting at its library default

def _r2(y, p):
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

def run_out_of_sample_cv(df: pd.DataFrame, n_splits: int = 5):
    print("\n>>> PHASE 6.4: OUT-OF-SAMPLE VALIDATION (EXPANDING-WINDOW TIME-SERIES CV)")
    path = os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV.csv")
    path_f = os.path.join(OUTPUT_DIR_P6, "Phase_6_4_Out_of_Sample_CV_Folds.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    d = df.dropna(subset=CV_FEATURES_AFRR + ['a-FRR_Price', 'System_Direction_Down']).copy()
    # NOTE: Phase 6.1 hands NaNs to XGBoost unchanged; the dataset has none (N = 35,064 in every
    # phase), so this row set is identical to the one the Phase 6.1 models were fitted on.
    if d['Solar_Generation_Share'].max() <= 1.0:        # Phase 6.1 works in percent
        d['Solar_Generation_Share'] *= 100
    tscv = TimeSeriesSplit(n_splits=n_splits)   # expanding window: train on the past, test on the future

    rows, folds = [], []
    try:
        for dep, label, feats in [('Mcp', 'MCP (price)', CV_FEATURES_CORE),
                                  ('a-FRR_Price', 'a-FRR (price)', CV_FEATURES_AFRR)]:
            X = d[feats].astype(float).values; y = d[dep].astype(float).values
            mae, rmse, r2 = [], [], []
            for k, (tr, te) in enumerate(tscv.split(X), 1):
                p = XGBRegressor(**CV_XGB_PARAMS).fit(X[tr], y[tr]).predict(X[te])
                mae.append(mean_absolute_error(y[te], p)); rmse.append(np.sqrt(mean_squared_error(y[te], p))); r2.append(_r2(y[te], p))
                folds.append({'Target': label, 'Fold': k, 'Train_N': len(tr), 'Test_N': len(te),
                              'Test_start': str(d.index[te[0]]), 'Test_end': str(d.index[te[-1]]),
                              'MAE': mae[-1], 'RMSE': rmse[-1], 'R2_oos': r2[-1], 'Accuracy': np.nan, 'F1': np.nan})
            r2_in = _r2(y, XGBRegressor(**CV_XGB_PARAMS).fit(X, y).predict(X))   # full-sample (Phase 6.1) model, in-sample
            rows.append({'Target': label, 'MAE': np.mean(mae), 'RMSE': np.mean(rmse), 'R2_oos': np.mean(r2),
                         'Accuracy': np.nan, 'F1': np.nan, 'R2_insample': r2_in, 'Accuracy_insample': np.nan})

        X = d[CV_FEATURES_CORE].astype(float).values; yc = d['System_Direction_Down'].astype(int).values
        acc, f1, auc, bacc, base = [], [], [], [], []
        for k, (tr, te) in enumerate(tscv.split(X), 1):
            clf = XGBClassifier(**CV_XGB_PARAMS).fit(X[tr], yc[tr])
            p = clf.predict(X[te]); pr = clf.predict_proba(X[te])[:, 1]
            acc.append(accuracy_score(yc[te], p)); f1.append(f1_score(yc[te], p))
            auc.append(roc_auc_score(yc[te], pr)); bacc.append(balanced_accuracy_score(yc[te], p))
            base.append(max(yc[te].mean(), 1 - yc[te].mean()))   # majority-class accuracy in the test fold
            # (share of the larger class in that held-out fold, i.e. an oracle 'always-majority' baseline;
            #  the surplus share drifts from ~0.19 to ~0.33 across folds, so the baseline is fold-specific)
            folds.append({'Target': 'System direction (classifier)', 'Fold': k, 'Train_N': len(tr), 'Test_N': len(te),
                          'Test_start': str(d.index[te[0]]), 'Test_end': str(d.index[te[-1]]),
                          'MAE': np.nan, 'RMSE': np.nan, 'R2_oos': np.nan, 'Accuracy': acc[-1], 'F1': f1[-1],
                          'ROC_AUC': auc[-1], 'Balanced_Acc': bacc[-1], 'Majority_baseline_acc': base[-1]})
        clf_full = XGBClassifier(**CV_XGB_PARAMS).fit(X, yc)
        acc_in = accuracy_score(yc, clf_full.predict(X))
        rows.append({'Target': 'System direction (classifier)', 'MAE': np.nan, 'RMSE': np.nan, 'R2_oos': np.nan,
                     'Accuracy': np.mean(acc), 'F1': np.mean(f1), 'R2_insample': np.nan, 'Accuracy_insample': acc_in,
                     'ROC_AUC': np.mean(auc), 'Balanced_Acc': np.mean(bacc), 'Majority_baseline_acc': np.mean(base)})

        out = pd.DataFrame(rows)
        pd.DataFrame(folds).to_csv(path_f, index=False, **CSV_KW)
        out.to_csv(path, index=False, **CSV_KW)
        print(out.to_string(index=False))
        print("\n  Note: metrics are averages over the 5 held-out folds (sklearn TimeSeriesSplit, expanding window).")
        print("        Models, features and hyper-parameters are identical to Phase 6.1 (the PDP models).")
        print("        The low F1 reflects the minority (surplus) class; XGBoost is an interpretive tool (PDP), not a forecasting model.")
        print("        For the imbalanced system-direction target, accuracy is compared with the majority-class baseline;")
        print("        ROC-AUC and balanced accuracy are the informative held-out measures.")
        print(f"  Saved: {path}\n  Saved: {path_f}")
    except Exception as e:
        print(f"  !! Error in out-of-sample validation: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_out_of_sample_cv(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")


## Phase 7: Robustness Checks (Quantile Regression)


In [ ]:
# =============================================================================
# PHASE 7: ROBUSTNESS CHECKS (QUANTILE REGRESSION) - CACHED
# =============================================================================

import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# [Revision] QR convergence warnings are shown, not silenced (the global setup ignores warnings)
from statsmodels.tools.sm_exceptions import IterationLimitWarning, ConvergenceWarning
warnings.simplefilter("always", IterationLimitWarning)
warnings.simplefilter("always", ConvergenceWarning)

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_7")
REPORT_FILE = os.path.join(OUTPUT_DIR, "Phase_7_Robustness_Quantile_Regression_Results.txt")
CSV_FILE = os.path.join(OUTPUT_DIR, "Phase_7_Quantile_Regression_Coefficients.csv")
PLOT_FILE = os.path.join(OUTPUT_DIR, "Phase_7_Figure_11_Quantile_Regression_Coefficient_Robustness.png")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def check_existing_results():
    """Checks if all required output files exist."""
    return os.path.exists(REPORT_FILE) and os.path.exists(CSV_FILE) and os.path.exists(PLOT_FILE)

def run_phase_7_exact_style(df):
    print("PHASE 7: Robustness Checks (Quantile Regression) Process Started...")
    print("="*80)

    # Cache check
    if check_existing_results():
        print(f">> [CACHE] Results found in '{OUTPUT_DIR}'. Loading...\n")

        with open(REPORT_FILE, "r", encoding="utf-8") as f:
            print(f.read())

        img = plt.imread(PLOT_FILE)
        plt.figure(figsize=(12, 6)); plt.imshow(img); plt.axis('off'); plt.show()

        print("="*80)
        print(f">> Results loaded from '{OUTPUT_DIR}' successfully.")
        return

    # Run quantile regression
    print(">> No cached results found. Running Quantile Regression (Q10, Q50, Q90)...")

    try:
        df_qr = df.copy()
        df_qr.rename(columns={'a-FRR_Price': 'aFRR_Price', 'Solar_Generation_Share': 'Solar_Share'}, inplace=True)

        controls = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                    'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                    'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
        control_str = " + ".join(controls)

        # [Revision_Review_Referee_2.5] convergence fix: standardise regressors
        reg_cols = ['Solar_Share'] + controls
        sd = df_qr[reg_cols].std()
        df_qr_s = df_qr.copy()
        df_qr_s[reg_cols] = df_qr_s[reg_cols] / sd      # regressors only; y stays in original units

        qs = [0.10, 0.50, 0.90]
        mcp_coefs, afrr_coefs, qr_results_data = [], [], []

        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            f.write("PHASE 7: QUANTILE REGRESSION ROBUSTNESS REPORT\n")
            f.write("="*80 + "\n\n")

            # H1: Merit-Order Effect (MCP)
            h1_title = ">>> HYPOTHESIS 1: MERIT-ORDER EFFECT ON MCP (ROBUSTNESS CHECK)\n"
            sep = "-" * 75 + "\n"
            header = f"{'Quantile':<10} | {'Coef (Solar)':<15} | {'P-Value':<10} | {'Status'}\n"

            f.write(h1_title + sep + header + sep)
            print("\n" + h1_title.strip())
            print(sep.strip())
            print(header.strip())
            print(sep.strip())

            formula_mcp = f"Mcp ~ Solar_Share + {control_str}"
            for q in qs:
                res = smf.quantreg(formula_mcp, df_qr_s).fit(q=q, max_iter=5000)
                coef = res.params['Solar_Share'] / sd['Solar_Share']   # back to original scale
                pval = res.pvalues['Solar_Share']
                mcp_coefs.append(coef)
                status = "CONFIRMED (-)" if (coef < 0 and pval < 0.05) else "Rejected"
                line = f"Q{int(q*100):<8} | {coef:>10.4f}      | {pval:.4f}     | {status}\n"
                print(line.strip()); f.write(line)
                qr_results_data.append({'Hypothesis': 'MCP', 'Quantile': q, 'Coef': coef, 'Pval': pval})

            # H2: Volatility Effect (a-FRR)
            h2_title = "\n\n>>> HYPOTHESIS 2: VOLATILITY EFFECT ON a-FRR (ROBUSTNESS CHECK)\n"
            header_v = f"{'Quantile':<10} | {'Coef (Solar)':<15} | {'P-Value':<10} | {'Interpretation'}\n"

            f.write(h2_title + sep + header_v + sep)
            print(h2_title.strip())
            print(sep.strip())
            print(header_v.strip())
            print(sep.strip())

            formula_afrr = f"aFRR_Price ~ Solar_Share + {control_str}"
            for q in qs:
                res = smf.quantreg(formula_afrr, df_qr_s).fit(q=q, max_iter=5000)
                coef = res.params['Solar_Share'] / sd['Solar_Share']   # back to original scale
                pval = res.pvalues['Solar_Share']
                afrr_coefs.append(coef)
                if q == 0.90 and coef > 5000: interp = "CRITICAL STRESS (!)"
                elif coef > 0 and pval < 0.05: interp = "Positive (+)"
                else: interp = "Insignificant"
                line = f"Q{int(q*100):<8} | {coef:>10.4f}      | {pval:.4f}     | {interp}\n"
                print(line.strip()); f.write(line)
                qr_results_data.append({'Hypothesis': 'aFRR', 'Quantile': q, 'Coef': coef, 'Pval': pval})

        # Visualization
        plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman']})
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
        categories = ['Q10 (Low)', 'Q50 (Median)', 'Q90 (High)']

        ax1.bar(categories, mcp_coefs, color='darkorange', alpha=0.7)
        ax1.axhline(0, color='black', linewidth=0.8)
        ax1.set_ylabel('Solar Impact Coefficient (MCP)')

        ax2.bar(categories, afrr_coefs, color='crimson', alpha=0.7)
        ax2.axhline(0, color='black', linewidth=0.8)
        ax2.set_ylabel('Solar Impact Coefficient (a-FRR)')

        plt.tight_layout()
        plt.savefig(PLOT_FILE, dpi=FIG_DPI, bbox_inches='tight')
        plt.suptitle("Phase 7 - Quantile Regression Robustness Checks (Preview)", fontsize=16, y=1.02)
        plt.show()

        pd.DataFrame(qr_results_data).to_csv(CSV_FILE, index=False)
        print(f"\n>> Results saved to '{OUTPUT_DIR}'.")

    except Exception as e:
        print(f"!! ERROR in Phase 7: {e}")

# --- Execution ---
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_phase_7_exact_style(target_df)

### Phase 7.2: Solver Iteration Sensitivity (Revision_Review_Referee_2.5)

Estimates the Phase 7 specification with statsmodels `QuantReg` on the unscaled regressors at 1,000 / 2,000 /
5,000 / 10,000 iterations, and re-runs the original Phase 7 call. Fits that stop at the iteration limit are flagged.

Produces: `Phase_7_2_QR_Iteration_Sensitivity.csv` in `results/Phase_7/`.

In [ ]:
# =============================================================================
# PHASE 7.2: QR SOLVER ITERATION SENSITIVITY - CACHED
# =============================================================================
# Revision_Review_Referee_2.5 — QR convergence diagnostics & dependence-robust SE
# Added in revision. Feeds Supplementary Appendix J.5.
# =============================================================================
import pandas as pd, numpy as np, os, warnings
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import IterationLimitWarning
# [Revision] QR convergence warnings are shown, not silenced (the global setup ignores warnings)
from statsmodels.tools.sm_exceptions import IterationLimitWarning, ConvergenceWarning
warnings.simplefilter("always", IterationLimitWarning)
warnings.simplefilter("always", ConvergenceWarning)

OUTPUT_DIR_P7 = os.path.join(RESULTS_BASE, "Phase_7")
os.makedirs(OUTPUT_DIR_P7, exist_ok=True)

# Same control set as Phase 7
QR_CONTROLS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
               'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
               'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
               'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
QUANTILES  = [0.10, 0.50, 0.90]
QR_TARGETS = [('Mcp', 'MCP'), ('a-FRR_Price', 'aFRR')]

# Reference values: submitted Table 1 (results/Phase_7) and first-draft Supplementary Table J5
TABLE_1_SUBMITTED = {('MCP', 0.10): -3980.01, ('MCP', 0.50): -5421.33, ('MCP', 0.90): -2716.16,
                     ('aFRR', 0.10): 1187.78, ('aFRR', 0.50): 5436.08, ('aFRR', 0.90): 11632.54}
TABLE_J5_DRAFT    = {('MCP', 0.10): -5011, ('MCP', 0.50): -4397, ('MCP', 0.90): -2711,
                     ('aFRR', 0.10): 1165, ('aFRR', 0.50): 4727, ('aFRR', 0.90): 11071}

def qr_design(df: pd.DataFrame):
    """Design matrix [const, solar, controls] and the std. deviation of each regressor."""
    Xr = df[['Solar_Generation_Share'] + QR_CONTROLS].astype(float)
    return sm.add_constant(Xr, has_constant='add').values, Xr.std().values

def check_loss(y, X, beta, q):
    """QR objective (sum of check-function losses); lower = better fit."""
    u = y - X @ beta
    return float(np.sum(u * (q - (u < 0))))

def fit_quantreg(y, X, q, max_iter):
    """statsmodels QuantReg fit that also reports whether the iteration limit was hit."""
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        res = sm.QuantReg(y, X).fit(q=q, max_iter=max_iter)
    converged = not any(issubclass(x.category, IterationLimitWarning) for x in w)
    return res, converged

def run_iteration_sensitivity(df: pd.DataFrame, iter_grid=(1000, 2000, 5000, 10000)):
    print("\n>>> PHASE 7.2: QR SOLVER ITERATION SENSITIVITY (UNSCALED REGRESSORS)")
    path = os.path.join(OUTPUT_DIR_P7, "Phase_7_2_QR_Iteration_Sensitivity.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    X, _ = qr_design(df)
    # original Phase 7 call (formula interface, hyphen-free names)
    df_qr = df.rename(columns={'a-FRR_Price': 'aFRR_Price', 'Solar_Generation_Share': 'Solar_Share'})
    control_str = " + ".join(QR_CONTROLS)

    rows = []
    try:
        print(f"  {'':10}" + "".join(f"{f'iter={mi}':>14}" for mi in iter_grid) + f"{'Phase7 rerun':>14}{'Table 1':>12}")
        for dep, label in QR_TARGETS:
            y = df[dep].astype(float).values
            dep_f = 'Mcp' if dep == 'Mcp' else 'aFRR_Price'
            for q in QUANTILES:
                row = {'Hypothesis': label, 'Quantile': q}
                for mi in iter_grid:
                    res, conv = fit_quantreg(y, X, q, mi)
                    row[f'Coef_iter{mi}'] = round(res.params[1], 1)
                    row[f'Converged_iter{mi}'] = conv
                row['Phase7_Code_Rerun'] = round(smf.quantreg(f"{dep_f} ~ Solar_Share + {control_str}", df_qr)
                                                 .fit(q=q, max_iter=2000).params['Solar_Share'], 1)
                row['Table_1_Submitted'] = TABLE_1_SUBMITTED[(label, q)]
                rows.append(row)
                cells = "".join(f"{row[f'Coef_iter{mi}']:>13,.1f}{' ' if row[f'Converged_iter{mi}'] else '*'}"
                                for mi in iter_grid)
                print(f"  {label:<5} Q{int(q*100):<3}{cells}{row['Phase7_Code_Rerun']:>14,.1f}{row['Table_1_Submitted']:>12,.1f}")

        out = pd.DataFrame(rows)
        out.to_csv(path, index=False)
        print("\n  * = iteration limit reached (solver did not converge).")
        print("  Note: a converged estimator would give the same coefficient in every column.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in iteration-sensitivity check: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_iteration_sensitivity(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

### Phase 7.3: Exact QR Solution (Linear Programming) (Revision_Review_Referee_2.5)

Computes the exact quantile-regression solution with the HiGHS linear-programming solver and compares it with
statsmodels IRLS on standardised regressors. The check-function loss identifies the minimising estimate.

Produces: `Phase_7_3_QR_Exact_Solution.csv` in `results/Phase_7/`.

In [ ]:
# =============================================================================
# PHASE 7.3: EXACT QR SOLUTION (LP / HiGHS) vs SCALED IRLS - CACHED
# =============================================================================
# Revision_Review_Referee_2.5 — QR convergence diagnostics & dependence-robust SE
# Added in revision. Feeds Supplementary Appendix J.5.
# =============================================================================
import pandas as pd, numpy as np, os, time
import statsmodels.api as sm
from sklearn.linear_model import QuantileRegressor

def run_exact_solution(df: pd.DataFrame):
    print("\n>>> PHASE 7.3: EXACT QR SOLUTION (LINEAR PROGRAMMING) vs SCALED IRLS")
    path = os.path.join(OUTPUT_DIR_P7, "Phase_7_3_QR_Exact_Solution.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    X, sd = qr_design(df)
    Xs = X.copy(); Xs[:, 1:] = X[:, 1:] / sd     # standardised regressors -> well-conditioned IRLS
    scale = np.r_[1.0, sd]                        # converts scaled coefficients back

    rows = []
    try:
        for dep, label in QR_TARGETS:
            y = df[dep].astype(float).values
            for q in QUANTILES:
                t0 = time.time()
                b_lp = QuantileRegressor(quantile=q, alpha=0, solver='highs',
                                         fit_intercept=False).fit(X, y).coef_
                t_lp = time.time() - t0
                res_s, conv_s = fit_quantreg(y, Xs, q, 5000)
                b_s = res_s.params / scale
                res_u, conv_u = fit_quantreg(y, X, q, 1000)   # statsmodels default, as in draft J5
                rows.append({'Hypothesis': label, 'Quantile': q,
                             'Coef_Exact_LP': round(b_lp[1], 1),
                             'Coef_Scaled_IRLS': round(b_s[1], 1),
                             'Coef_Unscaled_IRLS_1000': round(res_u.params[1], 1),
                             'Table_1_Submitted': TABLE_1_SUBMITTED[(label, q)],
                             'Loss_Exact_LP': round(check_loss(y, X, b_lp, q)),
                             'Loss_Scaled_IRLS': round(check_loss(y, X, b_s, q)),
                             'Loss_Unscaled_IRLS_1000': round(check_loss(y, X, res_u.params, q)),
                             'Scaled_IRLS_Converged': conv_s,
                             'Unscaled_IRLS_Converged': conv_u})
                r = rows[-1]
                print(f"  {label:<5} Q{int(q*100):<3} LP={r['Coef_Exact_LP']:>10,.1f}  "
                      f"scaled IRLS={r['Coef_Scaled_IRLS']:>10,.1f} (conv={conv_s})  "
                      f"unscaled IRLS={r['Coef_Unscaled_IRLS_1000']:>10,.1f} (conv={conv_u})  "
                      f"| loss LP={r['Loss_Exact_LP']:,} vs unscaled={r['Loss_Unscaled_IRLS_1000']:,}  [{t_lp:.0f}s]")

        out = pd.DataFrame(rows)
        out.to_csv(path, index=False)
        print("\n  Note: LP and scaled IRLS coincide and attain the lowest loss -> they are the QR estimates.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in exact QR solution: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_exact_solution(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

### Phase 7.4: Day-Clustered Block-Bootstrap Standard Errors (Revision_Review_Referee_2.5)

Re-estimates the quantile regressions with the converged solver and computes standard errors from a
day-clustered block bootstrap (whole 24-hour blocks resampled; 200 replications, seed 42).

Produces: `Phase_7_4_QR_Clustered_Bootstrap_SE.csv` in `results/Phase_7/`.

In [ ]:
# =============================================================================
# PHASE 7.4: DAY-CLUSTERED BLOCK-BOOTSTRAP QR STANDARD ERRORS - CACHED
# =============================================================================
# Revision_Review_Referee_2.5 — QR convergence diagnostics & dependence-robust SE
# Added in revision. Feeds Supplementary Appendix J.5.
# =============================================================================
import pandas as pd, numpy as np, os
import statsmodels.api as sm

N_BOOT_QR = 200
SEED_QR   = 42

def run_clustered_bootstrap(df: pd.DataFrame, n_boot: int = N_BOOT_QR, seed: int = SEED_QR):
    print("\n>>> PHASE 7.4: DAY-CLUSTERED BLOCK-BOOTSTRAP QR STANDARD ERRORS")
    path = os.path.join(OUTPUT_DIR_P7, "Phase_7_4_QR_Clustered_Bootstrap_SE.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    X, sd = qr_design(df)
    Xs = X.copy(); Xs[:, 1:] = X[:, 1:] / sd      # standardised regressors -> converged IRLS
    days = pd.to_datetime(df['Date']).dt.date.values
    uniq = np.unique(days)
    day_idx = {dd: np.where(days == dd)[0] for dd in uniq}
    rng = np.random.default_rng(seed)

    rows = []
    try:
        for dep, label in QR_TARGETS:
            y = df[dep].astype(float).values
            for q in QUANTILES:
                pt = sm.QuantReg(y, Xs).fit(q=q, max_iter=5000).params[1] / sd[0]
                bs = []
                for _ in range(n_boot):
                    sdays = rng.choice(uniq, size=len(uniq), replace=True)   # resample whole days
                    idx = np.concatenate([day_idx[dd] for dd in sdays])
                    try:
                        bs.append(sm.QuantReg(y[idx], Xs[idx]).fit(q=q, max_iter=5000).params[1] / sd[0])
                    except Exception:
                        pass
                se = float(np.std(bs)); t = pt / se if se else np.nan
                stars = '***' if abs(t) > 2.58 else ('**' if abs(t) > 1.96 else ('*' if abs(t) > 1.645 else ''))
                rows.append({'Hypothesis': label, 'Quantile': q, 'Coef': round(pt, 1),
                             'Clustered_SE': round(se, 1), 't_stat': round(t, 2), 'Sig': stars,
                             'Boot_OK': len(bs),
                             'Table_1_Submitted': TABLE_1_SUBMITTED[(label, q)],
                             'Table_J5_Draft': TABLE_J5_DRAFT[(label, q)]})
                print(f"  {label:<5} Q{int(q*100):<3} beta={pt:>10,.0f}  clustered-SE={se:>7,.0f}  t={t:>6.1f} {stars}")

        out = pd.DataFrame(rows)
        out.to_csv(path, index=False)
        print("\n" + out.to_string(index=False))
        print("\n  Note: *** p<0.01, ** p<0.05, * p<0.10.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in clustered bootstrap: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_clustered_bootstrap(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

## Phase 8: Non-Linear GAM Robustness (Cubic Spline)

Fits separate GAM models (OLS with natural cubic splines) for the surplus and deficit regimes to validate the non-linear flexibility-stress pattern identified in Phase 6, providing confidence intervals around the regime-specific a-FRR price trajectories.

In [ ]:
# =============================================================================
# PHASE 8: ROBUSTNESS CHECKS - NON-LINEAR GAM ANALYSIS (CUBIC SPLINE)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from patsy import cr
import warnings
import os

warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_8")
os.makedirs(OUTPUT_DIR, exist_ok=True)

REPORT_PATH = os.path.join(OUTPUT_DIR, "Phase_8_Flexibility_Stress_Report.txt")
PLOT_PATH = os.path.join(OUTPUT_DIR, "Phase_8_Figure_10_GAM_Flexibility_Stress.png")

plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],   # [MÖK-4] same typography as the other figures (was sans-serif)
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def execute_final_gam_analysis(df):
    print("PHASE 8: GAM Robustness Analysis (Cubic Spline)...")
    print("-" * 80)

    # Regime separation
    df_up_combined = df[df['System_Direction_Down'] == 0].copy()
    df_down = df[df['System_Direction_Down'] == 1].copy()

    print(f">> [Step 1/3] Data split: Deficit+Balanced ({len(df_up_combined)} obs), Surplus ({len(df_down)} obs)")
    print(">> [Step 2/3] Fitting OLS + Natural Cubic Splines models...")

    # Cache check
    if os.path.exists(REPORT_PATH) and os.path.exists(PLOT_PATH):
        print(f"\n>> [CACHE] Results found in '{OUTPUT_DIR}'. Loading...")

        with open(REPORT_PATH, "r", encoding="utf-8") as f:
            print(f.read())

        img = plt.imread(PLOT_PATH)
        plt.figure(figsize=(12, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Phase 8 - Figure 10: Non-Linear Flexibility-Stress Pattern (Preview - Loaded)", fontsize=14)
        plt.show()

        print(f">> Figure loaded: '{os.path.basename(PLOT_PATH)}'")
        return

    # Fit GAM models
    formula = 'Q("a-FRR_Price") ~ cr(Solar_Generation_Share, df=6) + Consumption + Wind_Generation + Import_Coal_Generation'
    model_up = smf.ols(formula, data=df_up_combined).fit()
    model_down = smf.ols(formula, data=df_down).fit()

    solar_grid = np.linspace(df['Solar_Generation_Share'].min(), df['Solar_Generation_Share'].max(), 100)
    pred_df = pd.DataFrame({
        'Solar_Generation_Share': solar_grid,
        'Consumption': df['Consumption'].mean(),
        'Wind_Generation': df['Wind_Generation'].mean(),
        'Import_Coal_Generation': df['Import_Coal_Generation'].mean()
    })

    res_up = model_up.get_prediction(pred_df).summary_frame(alpha=0.05)
    res_down = model_down.get_prediction(pred_df).summary_frame(alpha=0.05)

    # Visualization
    print(">> [Step 3/3] Generating figure...")
    plt.figure(figsize=(12, 7))
    solar_pct = solar_grid * 100

    plt.plot(solar_pct, res_up['mean'], color='green', lw=3, label='Energy Deficit & Balanced')
    plt.fill_between(solar_pct, res_up['mean_ci_lower'], res_up['mean_ci_upper'], color='green', alpha=0.15)

    plt.plot(solar_pct, res_down['mean'], color='red', lw=3, label='Energy Surplus')
    plt.fill_between(solar_pct, res_down['mean_ci_lower'], res_down['mean_ci_upper'], color='red', alpha=0.15)

    plt.xlabel("Solar Generation Share (%)")
    plt.ylabel("Simulated a-FRR Price (TL/MW)")  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW); plain text, not mathtext
    plt.axvline(x=20, color='grey', linestyle='--', alpha=0.6)
    plt.legend(loc='upper left', frameon=True, shadow=False)  # [Revision_Editor_E6] no shaded legend
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save clean, then preview with title
    plt.savefig(PLOT_PATH, dpi=FIG_DPI, bbox_inches='tight')
    plt.title("Phase 8 - Figure 10: Non-Linear Flexibility-Stress Pattern (Preview Only)", fontsize=14)
    plt.show()

    # Numerical analysis
    gap_start = res_up['mean'].iloc[0] - res_down['mean'].iloc[0]
    gap_end = res_up['mean'].iloc[-1] - res_down['mean'].iloc[-1]

    report_content = (
        "\n============================================================\n"
        "PHASE 8 NUMERICAL ANALYSIS RESULTS:\n"
        "------------------------------------------------------------\n"
        f"Starting Price Gap (0% Solar): {gap_start:.2f} TL\n"
        f"Ending Price Gap (Max Solar):  {gap_end:.2f} TL\n"
        f"Gap Expansion Factor:      {gap_end/gap_start:.2f}x\n"
        "============================================================"
    )

    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        f.write(report_content)

    print(report_content)
    print(f">> Figure saved: '{os.path.basename(PLOT_PATH)}'")

# --- Execution ---
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        execute_final_gam_analysis(target_df)
    else:
        print("Error: DataFrame 'df' not found. Please run Phase 1.")

### Phase 8.2: Statistical Identification of the Transition Onset (Revision_Review_Referee_2.4)

Estimates a **continuous regression-kink model** with an unknown threshold (Hansen, 2017) for the a-FRR price,

$$y_t = a + b\,s_t + \delta\,(s_t-\gamma)_+ + X_t'c + u_t,$$

where $s_t$ is the solar share (%), $(s_t-\gamma)_+ = \max(s_t-\gamma, 0)$ and $X_t$ is the manuscript's control
set. The threshold $\gamma$ is located by least squares over candidate solar shares of 5–44% (step 1 pp); it is a
*kink* (the slope changes from $b$ to $b+\delta$, the fitted line stays continuous), not the discontinuous
sample-splitting model of Hansen (2000). Reported: the 90% confidence interval for $\gamma$ from a day-clustered
bootstrap (whole days resampled; 200 replications, seed 42); the kink coefficient $\delta$ with its day-clustered
bootstrap standard error; and a sup-F test of $H_0{:}\ \delta = 0$ (no kink) whose p-value comes from a
day-clustered wild bootstrap under the null (Rademacher weights per day; 200 replications). This identifies the
~20% onset of balancing-cost acceleration. The higher ~33% figure is not estimated here: it is the surplus–deficit
crossover of the Phase 6.2 XGBoost interaction simulation.

Reference: Hansen, B. E. (2017). Regression kink with an unknown threshold. *Journal of Business & Economic
Statistics*, 35(2), 228–240. https://doi.org/10.1080/07350015.2015.1073595

Produces: `Phase_8_2_Hansen_Threshold.csv` in `results/Phase_8/`.


In [ ]:
# =============================================================================
# PHASE 8.2: REGRESSION-KINK THRESHOLD (HANSEN, 2017) + DAY-CLUSTERED BOOTSTRAP - CACHED
# =============================================================================
# Revision_Review_Referee_2.4 — Statistical identification of the ~20% threshold
# Added in revision. Feeds Supplementary Appendix J.4.
# Model: continuous regression kink  y = a + b*s + delta*(s - gamma)_+ + Xc*c
#        Hansen, B. E. (2017). Regression kink with an unknown threshold.
#        Journal of Business & Economic Statistics, 35(2), 228-240.
#        (NOT the discontinuous sample-splitting model of Hansen, 2000.)
# Outputs: gamma_hat (grid search); 90% CI from a day-clustered (whole-day) bootstrap;
#          delta (slope change at the kink) with day-clustered bootstrap SE and t;
#          sup-F test of H0: delta = 0 with a day-clustered wild-bootstrap p-value.
# NOTE: ~33% is NOT estimated here; it is the surplus/deficit crossover of the
#       Phase 6.2 XGBoost interaction simulation (confounder-controlled PDP).
# =============================================================================
import pandas as pd, numpy as np, os

THRESHOLD_CONTROLS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                      'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                      'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                      'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
N_BOOT_THR = 200
SEED_THR   = 42

def kink_design(s, Xc, g):
    """Design matrix of the continuous-kink model: [1, s, (s-g)_+, Xc]."""
    return np.column_stack([np.ones(len(s)), s, np.maximum(s - g, 0.0), Xc])

def kink_fit(y, s, Xc, g):
    """OLS of the kink model at threshold g -> (beta, SSR); beta = [a, b, delta, c...]."""
    X = kink_design(s, Xc, g)
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return beta, float(((y - X @ beta) ** 2).sum())

def kink_ssr(y, s, Xc, g):
    """SSR of the continuous-kink model y = a + b*s + d*(s-g)_+ + Xc*c."""
    return kink_fit(y, s, Xc, g)[1]

def kink_profile(y, s, Xc, grid):
    """Least-squares grid search -> (gamma_hat, beta_hat at gamma_hat, min SSR)."""
    ssr = np.array([kink_ssr(y, s, Xc, g) for g in grid])
    i = int(ssr.argmin())
    beta, _ = kink_fit(y, s, Xc, grid[i])
    return float(grid[i]), beta, float(ssr[i])

def run_threshold_regression(df: pd.DataFrame, dep: str = 'a-FRR_Price',
                             n_boot: int = N_BOOT_THR, seed: int = SEED_THR):
    print("\n>>> PHASE 8.2: REGRESSION-KINK THRESHOLD (HANSEN, 2017) + DAY-CLUSTERED BOOTSTRAP")
    path = os.path.join(OUTPUT_DIR, "Phase_8_2_Hansen_Threshold.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).T.to_string(header=False)); return

    grid = np.arange(5, 45, 1.0)   # candidate thresholds: 5%-44% solar share
    d = df.assign(solar_pct=df['Solar_Generation_Share'] * 100.0).dropna(subset=[dep, 'solar_pct'] + THRESHOLD_CONTROLS)
    if 'Date' not in d.columns:   # [Revision] Phase 5 normally adds df['Date']; keep this cell runnable on its own
        d = d.assign(Date=pd.DatetimeIndex(d.index).normalize())
    y, s = d[dep].astype(float).values, d['solar_pct'].values
    Xc = d[THRESHOLD_CONTROLS].astype(float).values
    n, k = len(y), Xc.shape[1] + 3            # k = parameters of the kink model
    try:
        # --- point estimates -------------------------------------------------
        ghat, beta_hat, ssr1 = kink_profile(y, s, Xc, grid)
        b_pre, delta = float(beta_hat[1]), float(beta_hat[2])
        # H0 (no kink): the same regression without the (s - gamma)_+ term
        X0 = np.column_stack([np.ones(n), s, Xc])
        b0, _, _, _ = np.linalg.lstsq(X0, y, rcond=None)
        fit0 = X0 @ b0; e0 = y - fit0; ssr0 = float((e0 ** 2).sum())
        supF = ((ssr0 - ssr1) / 1.0) / (ssr1 / (n - k))   # sup over the grid = F at gamma_hat

        days = pd.to_datetime(d['Date']).dt.date.values
        uniq, day_id = np.unique(days, return_inverse=True)
        day_idx = {dd: np.where(days == dd)[0] for dd in uniq}
        rng = np.random.default_rng(seed)

        # --- (i) day-clustered bootstrap of (gamma, delta): CI for gamma, SE for delta ---
        boot_g, boot_d = [], []
        for _ in range(n_boot):
            sdays = rng.choice(uniq, size=len(uniq), replace=True)   # resample whole days
            idx = np.concatenate([day_idx[dd] for dd in sdays])
            g_b, beta_b, _ = kink_profile(y[idx], s[idx], Xc[idx], grid)
            boot_g.append(g_b); boot_d.append(float(beta_b[2]))
        boot_g, boot_d = np.array(boot_g), np.array(boot_d)
        delta_se = float(boot_d.std(ddof=1)); delta_t = delta / delta_se if delta_se > 0 else np.nan

        # --- (ii) sup-F p-value: day-clustered wild bootstrap under H0 (Rademacher weight per day) ---
        supF_b = []
        for _ in range(n_boot):
            w = rng.choice([-1.0, 1.0], size=len(uniq))
            y_b = fit0 + w[day_id] * e0                          # null model + sign-flipped day blocks of residuals
            _, _, ssr1_b = kink_profile(y_b, s, Xc, grid)
            b0b, _, _, _ = np.linalg.lstsq(X0, y_b, rcond=None)
            ssr0_b = float(((y_b - X0 @ b0b) ** 2).sum())
            supF_b.append(((ssr0_b - ssr1_b) / 1.0) / (ssr1_b / (n - k)))
        supF_b = np.array(supF_b)
        p_supF = float((supF_b >= supF).mean())

        out = pd.DataFrame([{'Dependent': dep, 'Threshold_Point_Estimate_pct': ghat,
                             'CI90_Lower_pct': np.percentile(boot_g, 5), 'CI90_Upper_pct': np.percentile(boot_g, 95),
                             'Bootstrap_Median_pct': np.median(boot_g), 'N_Boot': n_boot, 'Seed': seed,
                             'Grid': '5-44% (step 1)', 'N': n,
                             'slope_pre_kink_b': b_pre, 'delta': delta, 'delta_se': delta_se, 'delta_t': delta_t,
                             'slope_post_kink_b_plus_delta': b_pre + delta,
                             'supF_stat': supF, 'supF_pval_bootstrap': p_supF, 'N_Boot_supF': n_boot,
                             'supF_boot_method': 'day-clustered wild bootstrap (Rademacher) under H0: delta = 0',
                             'Model': 'continuous regression kink, Hansen (2017) JBES 35(2)'}])
        out.to_csv(path, index=False, **CSV_KW)
        print(out.T.to_string(header=False))
        print(f"\n  gamma_hat = {ghat:.0f}%  90% CI [{np.percentile(boot_g, 5):.0f}%, {np.percentile(boot_g, 95):.0f}%]"
              f"  |  delta = {delta:,.1f} TL/MW per pp (boot SE {delta_se:,.1f}, t = {delta_t:.1f})"
              f"  |  sup-F = {supF:.1f}, bootstrap p = {p_supF:.3f} (resolution 1/{n_boot})")
        print("  Note: onset (~20%, this phase) and crossover (~33%, Phase 6.2 PDP) are complementary.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in threshold regression: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_threshold_regression(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")


## Phase 9: Methodological Framework Visualization

Generates a structured flowchart summarizing the entire research methodology, from data preparation through robust estimation and validation to final reporting.

In [ ]:
# =============================================================================
# PHASE 9: METHODOLOGICAL FRAMEWORK (WITH LOGISTIC REGRESSION)
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
import os

warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join(RESULTS_BASE, "Phase_9")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def generate_maximized_flowchart():
    print(f"Generating Methodological Framework (Saving to '{OUTPUT_DIR}')...")
    print("="*80)

    # Color palette
    COLOR_ORANGE = '#F58231'
    COLOR_BLUE_DARK = '#3A607E'
    COLOR_GREY = '#708090'
    COLOR_BLUE_LIGHT = '#9FD0E2'
    COLOR_WHITE = '#FFFFFF'
    COLOR_LINE = '#2C3E50'

    fig, ax = plt.subplots(figsize=(16, 22))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 22)
    ax.axis('off')

    # --- Helper: Section Banner ---
    # [Revision_Editor_E6] no background colour: white panel, section colour kept on frame and title
    def draw_banner(x, y, w, h, color, title):
        t_color = '#2E7FA0' if color == COLOR_BLUE_LIGHT else color   # light blue is too pale for text
        rect = patches.Rectangle((x, y), w, h, linewidth=4, edgecolor=t_color, facecolor=COLOR_WHITE, zorder=1)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h - 0.6, title.upper(), ha='center', va='center',
                fontsize=24, fontweight='bold', color=t_color, zorder=3)

    # --- Helper: Inner Box ---
    def draw_inner_box(x, y, w, h, text, font_size=18):
        rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor=COLOR_LINE, facecolor=COLOR_WHITE, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=font_size, wrap=True, zorder=3, linespacing=1.3, fontweight='medium')

    # --- Helper: Transition Arrows ---
    def draw_thick_transition(x, y_start, y_end):
        ax.annotate('', xy=(x, y_end), xytext=(x, y_start),
                    arrowprops=dict(arrowstyle='simple, head_width=1.5, head_length=1.5',
                                   facecolor=COLOR_LINE, edgecolor=COLOR_LINE, zorder=5))

    def draw_arrow(x_start, y_start, x_end, y_end):
        ax.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                    arrowprops=dict(arrowstyle='->', color=COLOR_LINE, lw=2.5, mutation_scale=10, zorder=4))

    # =============================================================================
    # ROW 1: DATA PREPARATION (Orange)
    # =============================================================================
    r1_y = 17.5
    draw_banner(0, r1_y, 14, 4.0, COLOR_ORANGE, "1. Data Preparation & Initial Modeling")

    box_y1 = r1_y + 0.5
    b_w = 2.4

    draw_inner_box(0.3, box_y1, b_w, 2.5, "Input:\nMCP, a-FRR,\nSolar, Cons.")
    draw_inner_box(3.0, box_y1, b_w, 2.5, "Feature Eng:\nSeasonality &\nInteractions")
    draw_inner_box(5.7, box_y1, b_w, 2.5, "Stationarity\nChecks (ADF)")
    draw_inner_box(8.4, box_y1, b_w, 2.5, "Initial Models\n(8 SARIMAX)")
    draw_inner_box(11.1, box_y1, b_w, 2.5, "Logistic\nRegression")

    draw_arrow(0.3 + b_w, box_y1 + 1.25, 3.0, box_y1 + 1.25)
    draw_arrow(3.0 + b_w, box_y1 + 1.25, 5.7, box_y1 + 1.25)
    draw_arrow(5.7 + b_w, box_y1 + 1.25, 8.4, box_y1 + 1.25)
    draw_arrow(8.4 + b_w, box_y1 + 1.25, 11.1, box_y1 + 1.25)

    draw_thick_transition(7.0, 17.5, 16.5)

    # =============================================================================
    # ROW 2: OPTIMIZATION (Dark Blue)
    # =============================================================================
    r2_y = 12.5
    draw_banner(0, r2_y, 14, 4.0, COLOR_BLUE_DARK, "2. Parameter Optimization & Robust Estimation")

    box_y2 = r2_y + 0.5
    draw_inner_box(1.0, box_y2, 3.5, 2.5, "Grid Search:\n216 Combinations\nper Model")
    draw_inner_box(5.25, box_y2, 3.5, 2.5, "Optimal Params\n(p,d,q)(P,D,Q)s")
    draw_inner_box(9.5, box_y2, 3.5, 2.5, "Robust Re-Est. &\nDiagnostic Checks")

    draw_arrow(4.5, box_y2 + 1.25, 5.25, box_y2 + 1.25)
    draw_arrow(8.75, box_y2 + 1.25, 9.5, box_y2 + 1.25)
    draw_thick_transition(7.0, 12.5, 11.5)

    # =============================================================================
    # ROW 3: CHECKS & VALIDATION (Grey)
    # =============================================================================
    r3_y = 7.5
    draw_banner(0, r3_y, 14, 4.0, COLOR_GREY, "3. Visual, ML & Robustness Checks")

    box_y3 = r3_y + 0.5
    draw_inner_box(0.5, box_y3, 3.0, 2.5, "Visual:\nIRF & Profile\nAnalysis")
    draw_inner_box(3.85, box_y3, 3.0, 2.5, "ML (XGBoost):\nNon-Linear\nPDP Curves")
    draw_inner_box(7.15, box_y3, 3.0, 2.5, "GAM Analysis:\nTransition Regime")
    draw_inner_box(10.5, box_y3, 3.0, 2.5, "Quantile Reg:\nDistributional\nValidation")

    # =============================================================================
    # ROW 4: REPORTING & FUTURE (Light Blue)
    # =============================================================================
    r4_y = 1.0
    draw_banner(0, r4_y, 6.5, 5.5, COLOR_BLUE_LIGHT, "4. Final Reporting")
    draw_inner_box(0.75, r4_y + 0.5, 5.0, 3.5,
                   "Supplementary Tables\nHeatmaps & Matrices\nFull Model Specs\nMethodological Report",
                   font_size=20)

    draw_banner(7.5, r4_y, 6.5, 5.5, COLOR_BLUE_LIGHT, "Future Extensions")
    rect_f = patches.Rectangle((8.0, r4_y + 0.5), 5.5, 3.5, linewidth=2, edgecolor=COLOR_LINE, facecolor=COLOR_WHITE, zorder=2)
    ax.add_patch(rect_f)
    future_text = "Battery Storage Integration\nPolicy Scenario Analysis\nCapacity Expansion Impact"
    ax.text(10.75, r4_y + 2.25, future_text, ha='center', va='center',
            fontsize=20, fontweight='medium', linespacing=2.0, zorder=3)

    draw_thick_transition(3.25, 7.5, 6.5)
    draw_thick_transition(10.75, 7.5, 6.5)

    # Save
    output_path = os.path.join(OUTPUT_DIR, 'Figure_3_Methodological_Framework_Updated.png')
    plt.savefig(output_path, dpi=FIG_DPI, bbox_inches='tight')

    print(f"\n>> Methodological framework saved to: '{output_path}'")
    plt.show()

if __name__ == "__main__":
    generate_maximized_flowchart()

## Phase 10: Appendix Generation (A-H)

Generates all supplementary appendices (A-F) for the study: descriptive statistics, ADF unit root tests, model selection comparison, full SARIMAX estimation results, XGBoost hyperparameters, and correlation heatmaps.

In [ ]:
# =============================================================================
# PHASE 10: APPENDIX GENERATION (APPENDIX A-H)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import warnings
import os

warnings.filterwarnings("ignore")

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

# Academic plot settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 8),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold'
})

def get_or_create_data() -> pd.DataFrame:
    """Loads 'df' from the global namespace or generates dummy data if missing."""
    target_df = globals().get('df', locals().get('df'))
    if target_df is None:
        print("!! WARNING: 'df' not found in memory. Please run Phase 1. Using dummy data for structure.")
        dates = pd.date_range(start='2022-01-01', periods=100, freq='H')
        target_df = pd.DataFrame({
            'Mcp': np.random.normal(2350, 800, 100),
            'a-FRR_Price': np.random.normal(1300, 1300, 100),
            'Solar_Generation_Share': np.random.uniform(0, 0.6, 100),
            'Consumption': np.random.normal(36000, 6000, 100),
            'System_Direction_Down': np.random.choice([0, 1], 100),
            'Wind_Generation': np.random.normal(5000, 2000, 100)
        }, index=dates)
    return target_df

def generate_appendix_a(df: pd.DataFrame):
    print("\n>>> APPENDIX A: DESCRIPTIVE STATISTICS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_A_Descriptive_Statistics.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path, index_col=0).to_string())
        return

    cols_to_desc = ['Mcp', 'a-FRR_Price', 'Solar_Generation_Share', 'Consumption', 'System_Direction_Down']
    if 'Natural_Gas_Generation' in df.columns:
        cols_to_desc.insert(2, 'Natural_Gas_Generation')

    valid_cols = [c for c in cols_to_desc if c in df.columns]
    desc_stats = df[valid_cols].describe().T[['mean', 'std', 'min', 'max', '50%']]
    desc_stats.columns = ['Mean', 'Std. Dev.', 'Min', 'Max', 'Median']

    desc_stats.to_csv(path)
    print(desc_stats.to_string())
    print(f"  Saved: {path}")

def generate_appendix_b(df: pd.DataFrame):
    print("\n>>> APPENDIX B: ADF UNIT ROOT TESTS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_B_ADF_Unit_Root_Tests.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    def run_adf(series: pd.Series, name: str) -> dict:
        result = adfuller(series.dropna())
        return {
            "Variable": name,
            "ADF Statistic": round(result[0], 4),
            "P-Value": round(result[1], 4),
            "Result": "I(0) - Stationary" if result[1] < 0.05 else "I(1) - Non-Stationary"
        }

    try:
        adf_results = []
        if 'Mcp' in df.columns: adf_results.append(run_adf(df['Mcp'], "MCP (Day-Ahead Price)"))
        if 'a-FRR_Price' in df.columns: 
            adf_results.append(run_adf(df['a-FRR_Price'], "a-FRR Price (Level)"))
            adf_results.append(run_adf(df['a-FRR_Price'].diff(), "a-FRR Price (1st Difference)"))
        if 'Solar_Generation_Share' in df.columns: adf_results.append(run_adf(df['Solar_Generation_Share'], "Solar Generation Share"))
        
        df_adf = pd.DataFrame(adf_results)
        df_adf.to_csv(path, index=False)
        print(df_adf.to_string(index=False))
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in ADF tests: {e}")

def generate_appendix_c():
    print("\n>>> APPENDIX C: MODEL SELECTION (AIC)")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_C_Model_Comparison.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    data_comparison = [
        {"Market": "MCP (Day-Ahead)", "Model": "Baseline", "Order": "(1,1,1)", "AIC": "10452.15", "Status": "Rejected"},
        {"Market": "MCP (Day-Ahead)", "Model": "Optimized", "Order": "(2,0,2)", "AIC": "9356.99", "Status": "Selected"},
        {"Market": "a-FRR (Balancing)", "Model": "Baseline", "Order": "(1,1,1)", "AIC": "10895.40", "Status": "Rejected"},
        {"Market": "a-FRR (Balancing)", "Model": "Optimized", "Order": "(1,1,2)", "AIC": "9350.88", "Status": "Selected"}
    ]
    df_compare = pd.DataFrame(data_comparison)
    df_compare.to_csv(path, index=False)
    print(df_compare.to_string(index=False))
    print(f"  Saved: {path}")

def generate_appendix_d():
    print("\n>>> APPENDIX D: FULL SARIMAX ESTIMATION RESULTS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_D_Full_SARIMAX_Results.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    sarimax_baseline = [
        {"Model": "Model 4 (MCP Baseline)", "Variable": "Solar_Generation_Share", "Coefficient": "-5152.06", "P-Value": "< 0.001", "Significance": "***"},
        {"Model": "Model 4 (MCP Baseline)", "Variable": "Consumption", "Coefficient": "+0.086", "P-Value": "< 0.001", "Significance": "***"},
        {"Model": "Model 1 (a-FRR Baseline)", "Variable": "Solar_Generation_Share", "Coefficient": "+3628.14", "P-Value": "< 0.001", "Significance": "***"},
        {"Model": "Model 1 (a-FRR Baseline)", "Variable": "Consumption", "Coefficient": "-0.081", "P-Value": "< 0.001", "Significance": "***"}
    ]
    df_d_full = pd.DataFrame(sarimax_baseline)
    df_d_full.to_csv(path, index=False)
    print(df_d_full.to_string(index=False))
    print(f"  Saved: {path}")

def generate_appendix_e():
    print("\n>>> APPENDIX E: XGBOOST HYPERPARAMETERS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_E_XGBoost_Params.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    # [Revision] Appendix E now lists the settings ACTUALLY used in Phases 6.1 / 6.2 / 6.4
    # (xgb.XGBRegressor / XGBClassifier(n_estimators=200, random_state=42); every other
    # setting at the XGBoost library default). No hyper-parameter tuning was performed:
    # XGBoost is an interpretive cross-check (partial-dependence plots), not a forecaster.
    xgb_params = {
        "Parameter": ["n_estimators", "learning_rate", "max_depth", "subsample", "colsample_bytree",
                      "objective (regressors)", "objective (classifier)", "random_state", "tuning"],
        "Value": [200, "0.3 (library default)", "6 (library default)", "1.0 (library default)",
                  "1.0 (library default)", "reg:squarederror", "binary:logistic", 42, "none (fixed a priori)"],
        "Rationale": ["Number of boosting rounds, fixed a priori", "Library default", "Library default",
                      "Library default (no row subsampling)", "Library default (no column subsampling)",
                      "Squared-error loss for MCP and a-FRR price", "Log-loss for system direction",
                      "Reproducibility (same seed in Phases 6.1, 6.2, 6.4)",
                      "Interpretive cross-check (PDP), not a tuned forecaster; out-of-sample fit in Phase 6.4"]
    }
    df_xgb = pd.DataFrame(xgb_params)
    df_xgb.to_csv(path, index=False, **CSV_KW)
    print(df_xgb.to_string(index=False))
    print(f"  Saved: {path}")

def generate_appendix_f(df: pd.DataFrame):
    print("\n>>> APPENDIX F: CORRELATION HEATMAPS")
    mcp_vars = ['Mcp', 'Consumption', 'Natural_Gas_Generation', 'Solar_Generation_Share', 'Wind_Generation', 'Import_Coal_Generation']
    afrr_vars = ['a-FRR_Price', 'Consumption', 'Natural_Gas_Generation', 'Solar_Generation_Share', 'Hydroelectric_Generation', 'System_Direction_Down']

    mcp_vars = [c for c in mcp_vars if c in df.columns]
    afrr_vars = [c for c in afrr_vars if c in df.columns]

    configs = [
        (mcp_vars, "Appendix_F1_MCP_Correlation", "MCP Market Correlations"),
        (afrr_vars, "Appendix_F2_aFRR_Correlation", "a-FRR Market Correlations")
    ]

    for vars_list, fname, title in configs:
        if len(vars_list) > 1:
            save_path = os.path.join(OUTPUT_DIR_P10, f"{fname}.png")
            
            if os.path.exists(save_path):
                print(f"  [CACHE] Loading {fname}...")
                plt.figure(figsize=(8, 6))
                plt.imshow(plt.imread(save_path))
                plt.axis('off')
                plt.show()
                continue

            try:
                plt.figure(figsize=(10, 8))
                corr = df[vars_list].corr()
                mask = np.triu(np.ones_like(corr, dtype=bool))
                sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
                plt.title(title, fontweight='bold')
                plt.tight_layout()
                plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight')
                plt.show()
                print(f"  Saved: {save_path}")
            except Exception as e:
                print(f"  !! Error generating heatmap {fname}: {e}")

def generate_appendix_g(df: pd.DataFrame):
    """Eksik olan Ek G: VIF (Multicollinearity) Testini uygular."""
    print("\n>>> APPENDIX G: VIF MULTICOLLINEARITY CHECKS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_G_VIF_Multicollinearity.csv")
    
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    cols_for_vif = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Import_Coal_Generation']
    valid_cols = [c for c in cols_for_vif if c in df.columns]
    
    if not valid_cols:
        print("  !! Required columns for VIF not found in dataframe.")
        return
        
    try:
        X = df[valid_cols].dropna()
        X_with_const = sm.add_constant(X)
        vif_data = pd.DataFrame()
        vif_data["Variable"] = X_with_const.columns
        vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
        
        vif_data = vif_data[vif_data["Variable"] != "const"].round(2)
        vif_data.to_csv(path, index=False)
        print(vif_data.to_string(index=False))
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error calculating VIF: {e}")

def generate_appendix_h(df: pd.DataFrame):
    print("\n>>> APPENDIX H: MANUAL DOWN-REGULATION VOLUME ANALYSIS")
    save_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_H_Down_Regulation_Volume.png')

    if os.path.exists(save_path):
        print(f"  [CACHE] Loading from {save_path}...")
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(save_path))
        plt.axis('off')
        plt.show()
        return

    dm = df.copy()
    if 'BPM_Down_Regulation_Volume' not in dm.columns:
        print("  !! 'BPM_Down_Regulation_Volume' column missing. Generating dummy data for visual structure...")
        dm['BPM_Down_Regulation_Volume'] = np.random.normal(500, 150, len(dm))

    if dm['Solar_Generation_Share'].max() <= 1:
        dm['Solar_Generation_Share'] *= 100

    dm['Solar_Regime'] = dm['Solar_Generation_Share'].apply(lambda x: '≥ 33%' if x >= 33 else '< 33%')

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(
        x='Solar_Regime', y='BPM_Down_Regulation_Volume', data=dm,
        order=['< 33%', '≥ 33%'], palette='coolwarm', ax=ax
    )
    
    ax.set_xlabel('Solar Penetration Regime', fontweight='bold')
    ax.set_ylabel('Manual Down-Regulation Volume (MWh)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {save_path}")

def run_phase_10(df):
    print(f">> PHASE 10: Appendix Generation (Directory: '{OUTPUT_DIR_P10}')...")
    print("="*80)
    
    try:
        generate_appendix_a(df)
        generate_appendix_b(df)
        generate_appendix_c()
        generate_appendix_d()
        generate_appendix_e()
        generate_appendix_f(df)
        # generate_appendix_g(df)   # [Revision] retired: VIFs are reported in Phase 11.2 -> Appendix J.1
        generate_appendix_h(df)

        print("\n" + "="*80)
        print(f"PHASE 10 COMPLETE. All Appendices (A-H) saved to '{OUTPUT_DIR_P10}'.")
        print("="*80)

    except Exception as e:
        print(f"\n!! CRITICAL PIPELINE ERROR: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    df_target = get_or_create_data()
    run_phase_10(df_target)

### Otantik Yeniden Hesap Anahtarı

`FORCE_FRESH = True` iken aşağıdaki dört analizin (ADF & KPSS, Phase 12, XGBoost ve GAM
metrikleri) cache dosyaları silinir ve sonuçlar bir sonraki hücre çalıştırmalarında
**sıfırdan, bu bilgisayardaki veriden** hesaplanır. Sayılar kesinleşip Word
güncellendikten sonra `False` yaparak cache'i tekrar devreye al. Faz 2–11
cache'lerine dokunmaz.

Ayrıca eski adlandırmadan kalan çıktıları (dergiye özel `Scissor` etiketleri, `Phase_11_Botas` ve `Phase_12_Inflation` klasörleri) tek seferlik temizler; temizlik `FORCE_FRESH` değerinden bağımsız her koşuda kontrol edilir ve yapılacak iş yoksa sessiz kalır.

In [ ]:
# =============================================================================
# OTANTİK YENİDEN HESAP ANAHTARI (CACHE TEMİZLEME)
# =============================================================================
# FORCE_FRESH = True  -> dört analiz hücresinin cache dosyaları silinir; sonuçlar
#                        SIFIRDAN, bu bilgisayardaki veriden hesaplanır.
# FORCE_FRESH = False -> cache korunur, kayıtlı sonuçlar okunur.
import os

FORCE_FRESH = False

if 'RESULTS_BASE' not in globals():
    RESULTS_BASE = "results" if os.path.isdir("results") else os.path.join("..", "results")

_fresh_targets = [
    os.path.join(RESULTS_BASE, "Phase_10", "Appendix_B_Stationarity_Tests.csv"),
    os.path.join(RESULTS_BASE, "Phase_11_Inflation", "Inflation_Robustness_Results.csv"),
    os.path.join(RESULTS_BASE, "Supplementary_Metrics", "XGBoost_Performance_Metrics.csv"),
    os.path.join(RESULTS_BASE, "Supplementary_Metrics", "GAM_Performance_Metrics.csv"),
]
if FORCE_FRESH:
    print(">> OTANTİK MOD: dört analiz cache'i temizleniyor, sonuçlar sıfırdan hesaplanacak.")
    for p in _fresh_targets:
        if os.path.exists(p):
            os.remove(p)
            print("   Silindi:", p)
        else:
            print("   Zaten yok:", p)
else:
    print(">> FORCE_FRESH = False — cache korunuyor, kayıtlı sonuçlar kullanılacak.")

# --- Eski adlandırmadan kalan çıktıların tek seferlik temizliği (idempotent) ---
import shutil
_stale = [
    os.path.join(RESULTS_BASE, "Phase_11_Botas"),                                 # Appendix I artık Phase_10 altında
    os.path.join(RESULTS_BASE, "Phase_12_Inflation"),                             # Phase_11_Inflation oldu
    os.path.join(RESULTS_BASE, "Phase_8", "Phase_8_Scissor_Effect_Report.txt"),   # yeni ad: Flexibility_Stress
    os.path.join(RESULTS_BASE, "Phase_8", "Phase_8_Figure_10_Final_Scissor_Effect_GAM.png"),
    os.path.join(RESULTS_BASE, "Phase_8", "Phase_8_Figure_10_Final_Flexibility_Trap_GAM.png"),  # ara adlandırma turundan kalan
]
for p in _stale:
    if os.path.isdir(p):
        shutil.rmtree(p); print("   Eski klasör silindi:", p)
    elif os.path.exists(p):
        os.remove(p); print("   Eski dosya silindi:", p)

# --- Phase 4 cache'indeki eski model etiketini düzelt (yalnızca metin; sayılar değişmez) ---
for _f in ["Phase4_Full_Estimation_Report.txt", "Phase4_All_Coefficients.csv"]:
    p = os.path.join(RESULTS_BASE, "Phase_4", _f)
    if os.path.exists(p):
        _t = open(p, encoding="utf-8", errors="ignore").read()
        if "Scissor Effect" in _t:
            open(p, "w", encoding="utf-8").write(_t.replace("Scissor Effect", "Flexibility Trap"))
            print("   Etiket güncellendi (Scissor Effect -> Flexibility Trap):", p)

# Not: FORCE_FIGURES anahtarı kurulum hücresine (en üst) taşındı —
# figür fazları bu hücreden önce koştuğu için silme işlemi orada yapılmalı.


### Phase 10 (Extension): Appendix B — ADF & KPSS Stationarity Tests

Extends Appendix B with joint ADF & KPSS stationarity testing (manuscript Table B1).
Saves `Appendix_B_Stationarity_Tests.csv` to `results/Phase_10/`.

In [ ]:
# =============================================================================
# PHASE 10 (EXTENSION): APPENDIX B - ADF & KPSS STATIONARITY TESTS (CACHED)
# =============================================================================
import pandas as pd
import os
import warnings
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

def generate_appendix_b_stationarity(df: pd.DataFrame):
    print("\n>>> APPENDIX B: ADF & KPSS STATIONARITY TESTS")
    path = os.path.join(OUTPUT_DIR_P10, "Appendix_B_Stationarity_Tests.csv")

    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    def run_tests(series: pd.Series, name: str) -> dict:
        s = series.dropna()
        adf = adfuller(s)                                              # H0: unit root (non-stationary)
        kpss_stat, kpss_p, _, kpss_crit = kpss(s, regression='c', nlags='auto')  # H0: stationary
        adf_stationary  = adf[1] < 0.05                               # reject unit root
        kpss_stationary = kpss_stat < kpss_crit['5%']                 # fail to reject stationarity
        if adf_stationary and kpss_stationary:
            verdict = "I(0) - Stationary (both)"
        elif (not adf_stationary) and (not kpss_stationary):
            verdict = "I(1) - Non-Stationary (both)"
        else:
            verdict = "Long memory / persistence (tests diverge)"
        return {
            "Variable": name,
            "ADF Statistic": round(adf[0], 4), "ADF P-Value": round(adf[1], 4),
            "KPSS Statistic": round(kpss_stat, 4), "KPSS P-Value": round(kpss_p, 4),
            "KPSS 5% Crit.": round(kpss_crit['5%'], 4),
            "Conclusion": verdict
        }

    try:
        results = []
        if 'Mcp' in df.columns: results.append(run_tests(df['Mcp'], "MCP (Day-Ahead Price)"))
        if 'a-FRR_Price' in df.columns:
            results.append(run_tests(df['a-FRR_Price'], "a-FRR Price (Level)"))
            results.append(run_tests(df['a-FRR_Price'].diff(), "a-FRR Price (1st Difference)"))
        if 'Solar_Generation_Share' in df.columns: results.append(run_tests(df['Solar_Generation_Share'], "Solar Generation Share"))

        df_out = pd.DataFrame(results)
        df_out.to_csv(path, index=False)
        print(df_out.to_string(index=False))
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in stationarity tests: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        generate_appendix_b_stationarity(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

## Phase 10 (Appendix I): Robustness Check with Natural Gas Cost Control

Re-estimates the Flexibility Trap (XGBoost) and GAM flexibility-stress models with the
Botaş natural-gas tariff as an additional control variable, using the extended dataset
(`Model_Data.xlsx`).

Produces **Appendix I** under `results/Phase_10/`: **Figure I.1** (Flexibility Trap +
Botaş Tariff) and **Figure I.2** (GAM Flexibility-Stress + Botaş Tariff).

In [ ]:
# =============================================================================
# PHASE 10 — APPENDIX I: ROBUSTNESS CHECK WITH NATURAL GAS COST CONTROL (BOTAŞ)
# =============================================================================
# Produces: Appendix I — Figure I.1 (Flexibility Trap + Botaş), Figure I.2 (GAM Flexibility-Stress + Botaş)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import statsmodels.formula.api as smf
from patsy import cr
import os
import warnings

warnings.filterwarnings("ignore")

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

# Orijinal akademik stile en yakın ayarlar
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

def run_appendix_i_botas(df_b):
    print(f"\n>> PHASE 10 — APPENDIX I: Botaş Robustness Check (Directory: '{OUTPUT_DIR_P10}')...")
    print("="*80)

    if 'Botas_Tariff' not in df_b.columns:
        print("!! WARNING: 'Botas_Tariff' column not found in dataset. Skipping Appendix I.")
        return

    fig11_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Figure_I1_Flexibility_Trap_Botas.png')
    fig12_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png')
    report_path = os.path.join(OUTPUT_DIR_P10, 'Appendix_I_Botas_Report.txt')

    # =========================================================================
    # 1. XGBOOST FLEXIBILITY TRAP (+ BOTAS_TARIFF)
    # =========================================================================
    if os.path.exists(fig11_path):
        print(f"  [CACHE] Loading {os.path.basename(fig11_path)}...")
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(fig11_path))
        plt.axis('off')
        plt.show()
    else:
        print("  [Fig 11] Training XGBoost with Botaş Tariff...")
        dm = df_b.copy()
        if dm['Solar_Generation_Share'].max() <= 1:
            dm['Solar_Generation_Share'] *= 100

        sur = dm[dm['System_Direction_Down'] == 1]
        dfc = dm[dm['System_Direction_Down'] == 0]

        ft = ['Solar_Generation_Share','Mcp','Consumption','Wind_Generation','Hour_Sin','Hour_Cos','Botas_Tariff']

        ms = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(sur[ft], sur['a-FRR_Price'])
        md = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(dfc[ft], dfc['a-FRR_Price'])

        sr = np.linspace(0, 60, 100)
        avg = dm[ft].median()
        sim = pd.DataFrame([avg]*100, columns=ft)
        sim['Solar_Generation_Share'] = sr

        mk = sr <= 45
        yd = md.predict(sim[mk][ft])
        ys = ms.predict(sim[ft])
        ysc = ys[:len(yd)]
        sc = sr[mk]

        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.plot(sc, yd, color='#1f77b4', lw=3.0, label='Energy Deficit & Balanced')
        ax.plot(sr, ys, color='red', lw=3.0, label='Energy Surplus')
        
        ax.fill_between(sc, ysc, yd, where=(yd>=ysc), interpolate=True, color='gray', alpha=.15, hatch='...', label='Elasticity Risk Gap')
        ax.fill_between(sc, ysc, yd, where=(ysc>yd), interpolate=True, color='red', alpha=.10, hatch='xxx', label='Flexibility Trap Zone')
        
        ax.set_xlabel('Solar Generation Share (%)')
        ax.set_ylabel('Predicted a-FRR Price (TL/MW)')  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)
        
        ax.legend(loc='upper left', fontsize=11, frameon=True)
        ax.grid(True, ls='--', alpha=.4)
        ax.axvline(x=45, color='gray', ls=':', alpha=.5)
        
        plt.tight_layout()
        plt.savefig(fig11_path, dpi=FIG_DPI, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {fig11_path}")

    # =========================================================================
    # 2. GAM FLEXIBILITY-STRESS (+ BOTAS_TARIFF)
    # =========================================================================
    if os.path.exists(fig12_path) and os.path.exists(report_path):
        print(f"  [CACHE] Loading {os.path.basename(fig12_path)} and Report...")
        plt.figure(figsize=(10, 6))
        plt.imshow(plt.imread(fig12_path))
        plt.axis('off')
        plt.show()
        
        with open(report_path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("\n  [Fig 12] Training GAM Flexibility-Stress with Botaş Tariff...")
        
        du = df_b[df_b['System_Direction_Down'] == 0]
        dd = df_b[df_b['System_Direction_Down'] == 1]

        formula = 'Q("a-FRR_Price") ~ cr(Solar_Generation_Share,df=6)+Consumption+Wind_Generation+Import_Coal_Generation+Botas_Tariff'

        mu = smf.ols(formula, data=du).fit()
        md = smf.ols(formula, data=dd).fit()

        sg = np.linspace(df_b['Solar_Generation_Share'].min(), df_b['Solar_Generation_Share'].max(), 100)
        pred_df = pd.DataFrame({
            'Solar_Generation_Share': sg,
            'Consumption': df_b['Consumption'].mean(),
            'Wind_Generation': df_b['Wind_Generation'].mean(),
            'Import_Coal_Generation': df_b['Import_Coal_Generation'].mean(),
            'Botas_Tariff': df_b['Botas_Tariff'].mean()
        })

        ru = mu.get_prediction(pred_df).summary_frame(alpha=.05)
        rd = md.get_prediction(pred_df).summary_frame(alpha=.05)
        
        # Yalnızca x-ekseni (çizim) için % ile çarpıyoruz
        sp = sg * 100

        fig, ax = plt.subplots(figsize=(10, 6))
        
        
        ax.plot(sp, ru['mean'], color='#1f77b4', lw=3.0, label='Energy Deficit & Balanced')
        ax.fill_between(sp, ru['mean_ci_lower'], ru['mean_ci_upper'], color='#1f77b4', alpha=.12)
        ax.plot(sp, rd['mean'], color='red', lw=3.0, label='Energy Surplus')
        ax.fill_between(sp, rd['mean_ci_lower'], rd['mean_ci_upper'], color='red', alpha=.12)
        
        ax.set_xlabel('Solar Generation Share (%)')
        ax.set_ylabel('Simulated a-FRR Price (TL/MW)')  # [Revision_Review_Referee_2.7] a-FRR is a capacity price (TL/MW)
        
        ax.axvline(x=20, color='grey', ls='--', alpha=.5)
        ax.legend(loc='upper left', fontsize=11, frameon=True)
        ax.grid(True, alpha=.3)
        
        plt.tight_layout()
        plt.savefig(fig12_path, dpi=FIG_DPI, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {fig12_path}")

        # Save and Print R² Report
        report_content = f"=== Appendix I: GAM Flexibility-Stress (Botaş Controlled) ===\n" \
                         f"Deficit/Balanced R²: {mu.rsquared:.4f}\n" \
                         f"Surplus R²: {md.rsquared:.4f}\n"
                         
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        print(f"\n{report_content}")
        print(f"  Report Saved: {report_path}")

    print("\n>> Appendix I (Botaş) complete.")
    print("="*80)

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = globals().get('df', locals().get('df'))
    
    if target_df is not None and 'Botas_Tariff' in target_df.columns:
        run_appendix_i_botas(target_df)
    else:
        botas_data_path = os.path.join(DATA_DIR, 'Model_Data.xlsx')
        print(f">> Loading extended dataset from {botas_data_path}...")
        try:
            df_botas = pd.read_excel(botas_data_path)
            if 'Date_Hour' in df_botas.columns:
                df_botas['Date_Hour'] = pd.to_datetime(df_botas['Date_Hour'])
                df_botas.set_index('Date_Hour', inplace=True)
                df_botas.sort_index(inplace=True)
            run_appendix_i_botas(df_botas)
        except Exception as e:
            print(f"!! FATAL ERROR: {e}")

## Phase 11: Inflation-Robustness Checks (Ratio & Cap-Normalised FE-OLS)

Verifies that the MCP / a-FRR divergence is not an artefact of inflation or currency
drift (manuscript Section 4.6.3). Estimates fixed-effects OLS models
(Hour + Month + Year FE, standard errors clustered by day) for:
the a-FRR/MCP normalisation ratio and cap-normalised MCP & a-FRR prices.

Produces: `Inflation_Robustness_Results.csv` in `results/Phase_11_Inflation/`.

In [ ]:
# =============================================================================
# PHASE 11: INFLATION-ROBUSTNESS CHECKS (RATIO & CAP-NORMALISED FE-OLS) - CACHED
# =============================================================================
import pandas as pd, numpy as np, os, warnings
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

OUTPUT_DIR_P11 = os.path.join(RESULTS_BASE, "Phase_11_Inflation")
os.makedirs(OUTPUT_DIR_P11, exist_ok=True)

def generate_inflation_robustness(df: pd.DataFrame):
    print("\n>>> PHASE 11: INFLATION-ROBUSTNESS (RATIO & CAP-NORMALISED FE-OLS)")
    path = os.path.join(OUTPUT_DIR_P11, "Inflation_Robustness_Results.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    d = df.copy()
    # statsmodels formulas cannot contain hyphens -> rename
    d = d.rename(columns={'a-FRR_Price': 'aFRR',
                          'a-FRR_Price_Divide_Price_Cap': 'aFRR_over_Cap',
                          'Mcp_Divide_Price_Cap': 'Mcp_over_Cap'})
    d['Date'] = pd.to_datetime(d['Date']); d['Year'] = d['Date'].dt.year

    # inflation-neutral ratio (drop MCP<1 to avoid blow-up; winsorise 1/99)
    d_ratio = d[d['Mcp'] >= 1].copy()
    d_ratio['Ratio'] = d_ratio['aFRR'] / d_ratio['Mcp']
    lo, hi = d_ratio['Ratio'].quantile([.01, .99]); d_ratio['Ratio'] = d_ratio['Ratio'].clip(lo, hi)

    CTRL = ("Consumption + Wind_Generation_Share + Hydroelectric_Generation_Share "
            "+ Botas_Tariff + Weekend + Public_Holiday + C(Hour) + C(Month) + C(Year)")

    def fe_ols(dep, data, label):
        m = smf.ols(f"{dep} ~ Solar_Generation_Share + {CTRL}", data=data)\
               .fit(cov_type='cluster', cov_kwds={'groups': data['Date']})
        b = m.params['Solar_Generation_Share']
        return {"Dependent Variable": label,
                "Solar Coef (per unit share)": round(b, 4),
                "Effect per +1pp": round(b/100, 4),
                "Std. Error": round(m.bse['Solar_Generation_Share'], 4),
                "P-Value": round(m.pvalues['Solar_Generation_Share'], 4),
                "R-squared": round(m.rsquared, 3),
                "N": int(m.nobs)}
    try:
        rows = [
            fe_ols("Mcp",           d,       "MCP (level)"),        # merit-order, reference
            fe_ols("aFRR",          d,       "a-FRR (level)"),      # balancing, reference
            fe_ols("Ratio",         d_ratio, "a-FRR / MCP ratio"),  # inflation-neutral
            fe_ols("Mcp_over_Cap",  d,       "MCP / Price Cap"),    # inflation-neutral
            fe_ols("aFRR_over_Cap", d,       "a-FRR / Price Cap"),  # inflation-neutral
        ]
        out = pd.DataFrame(rows)
        out.to_csv(path, index=False)
        print(out.to_string(index=False))
        print("\n  Note: Hour+Month+Year fixed effects; std. errors clustered by day.")
        print(f"  Ratio model drops {len(d)-len(d_ratio)} hours with MCP<1 and winsorises at 1/99 pct.")
        print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in inflation-robustness checks: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        generate_inflation_robustness(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

In [ ]:
# =============================================================================
# SUPPLEMENTARY: XGBOOST PERFORMANCE METRICS (R² AND ACCURACY) - CACHED
# =============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score, accuracy_score
import os
import warnings

warnings.filterwarnings("ignore")

OUTPUT_DIR_SUPP = os.path.join(RESULTS_BASE, "Supplementary_Metrics")
os.makedirs(OUTPUT_DIR_SUPP, exist_ok=True)

def run_xgboost_performance_metrics():
    print(">> Computing XGBoost performance metrics...")
    path = os.path.join(OUTPUT_DIR_SUPP, "XGBoost_Performance_Metrics.csv")

    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        return

    # 1. Load data
    df_ml = None
    try:
        df_ml = pd.read_excel(DATA_FILE)
        print(f">> '{DATA_FILE}' loaded successfully.")
    except Exception:
        if 'df' in globals() and globals()['df'] is not None:
            df_ml = globals()['df'].copy()
            print(">> Using in-memory 'df'.")
        else:
            print("ERROR: Dataset not found. Ensure the data file exists or run Phase 1.")
            return

    # 2. Data preparation
    if df_ml['Solar_Generation_Share'].max() <= 1.0:
        df_ml['Solar_Generation_Share'] *= 100

    features_core = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Price_Cap', 'Hour_Sin', 'Hour_Cos']
    features_with_mcp = features_core + ['Mcp']
    # Phase 6.2 (Flexibility Trap) ile birebir aynı set — rejim modelleri bu setle ölçülmeli
    features_trap = ['Solar_Generation_Share', 'Mcp', 'Consumption', 'Wind_Generation', 'Hour_Sin', 'Hour_Cos']

    # 3. General models (Phase 6.1)
    print(">> Training general models...")

    m_a = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['Mcp'])
    m_b = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(df_ml[features_with_mcp], df_ml['a-FRR_Price'])
    m_c = xgb.XGBClassifier(n_estimators=200, random_state=42).fit(df_ml[features_core], df_ml['System_Direction_Down'])

    # 4. Regime-conditional models (Phase 6.2 - Flexibility Trap)
    print(">> Training regime-conditional models...")

    surplus_data = df_ml[df_ml['System_Direction_Down'] == 1]
    deficit_bal_data = df_ml[df_ml['System_Direction_Down'] == 0]

    model_surplus = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(surplus_data[features_trap], surplus_data['a-FRR_Price'])
    model_deficit = xgb.XGBRegressor(n_estimators=200, random_state=42).fit(deficit_bal_data[features_trap], deficit_bal_data['a-FRR_Price'])

    # 5. Metrics
    y_pred_mcp = m_a.predict(df_ml[features_core])
    r2_mcp = r2_score(df_ml['Mcp'], y_pred_mcp)

    y_pred_afrr = m_b.predict(df_ml[features_with_mcp])
    r2_afrr = r2_score(df_ml['a-FRR_Price'], y_pred_afrr)

    y_pred_dir = m_c.predict(df_ml[features_core])
    acc_dir = accuracy_score(df_ml['System_Direction_Down'], y_pred_dir)

    y_pred_def = model_deficit.predict(deficit_bal_data[features_trap])
    r2_def = r2_score(deficit_bal_data['a-FRR_Price'], y_pred_def)

    y_pred_sur = model_surplus.predict(surplus_data[features_trap])
    r2_sur = r2_score(surplus_data['a-FRR_Price'], y_pred_sur)

    # 6. Save + report
    out = pd.DataFrame([
        {"Metric": "MCP Model R2 Score",            "Value": round(r2_mcp, 3)},
        {"Metric": "a-FRR (Volatility) R2 Score",   "Value": round(r2_afrr, 3)},
        {"Metric": "System Direction Accuracy (%)", "Value": round(acc_dir * 100, 1)},
        {"Metric": "Deficit Regime Model R2",       "Value": round(r2_def, 3)},
        {"Metric": "Surplus Regime Model R2",       "Value": round(r2_sur, 3)},
    ])
    out.to_csv(path, index=False)

    print("\n" + "=" * 50)
    print("XGBOOST PERFORMANCE METRICS SUMMARY")
    print("=" * 50)
    print(f"1. MCP Model R2 Score:             {r2_mcp:.3f}")
    print(f"2. a-FRR (Volatility) R2 Score:    {r2_afrr:.3f}")
    print(f"3. System Direction Accuracy:      {acc_dir*100:.1f}%")
    print(f"4. Deficit Regime Model R2:        {r2_def:.3f}")
    print(f"5. Surplus Regime Model R2:        {r2_sur:.3f}")
    print("=" * 50)
    print(f"  Saved: {path}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    run_xgboost_performance_metrics()

In [ ]:
# =============================================================================
# SUPPLEMENTARY: GAM PERFORMANCE METRICS (R²) - CACHED
# =============================================================================

import pandas as pd
import os
import statsmodels.formula.api as smf
from patsy import cr

OUTPUT_DIR_SUPP = os.path.join(RESULTS_BASE, "Supplementary_Metrics")
os.makedirs(OUTPUT_DIR_SUPP, exist_ok=True)

def check_gam_performance(df):
    print("\n>> Computing GAM model R-squared values...")
    print("-" * 50)
    path = os.path.join(OUTPUT_DIR_SUPP, "GAM_Performance_Metrics.csv")

    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False))
        print("-" * 50)
        return

    # Regime separation
    df_up = df[df['System_Direction_Down'] == 0].copy()
    df_down = df[df['System_Direction_Down'] == 1].copy()

    # Same formula as Phase 8
    formula = 'Q("a-FRR_Price") ~ cr(Solar_Generation_Share, df=6) + Consumption + Wind_Generation + Import_Coal_Generation'

    m_up = smf.ols(formula, data=df_up).fit()
    m_down = smf.ols(formula, data=df_down).fit()

    out = pd.DataFrame([
        {"Model": "Energy DEFICIT (Up-Regulation)",   "R-squared": round(m_up.rsquared, 4),   "N": int(m_up.nobs)},
        {"Model": "Energy SURPLUS (Down-Regulation)", "R-squared": round(m_down.rsquared, 4), "N": int(m_down.nobs)},
    ])
    out.to_csv(path, index=False)

    print(f"1. Energy DEFICIT Model (Up-Regulation) R-squared: {m_up.rsquared:.4f}")
    print(f"2. Energy SURPLUS Model (Down-Regulation) R-squared: {m_down.rsquared:.4f}")
    print(f"  Saved: {path}")
    print("-" * 50)

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        check_gam_performance(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

### Phase 11.2: Timing, Decision-Time Information & Confounding (Revision_Review_Referee_2.1)

Re-estimates the solar coefficients with information available at each market's decision time: the solar share
lagged one day (D−1) for the day-ahead MCP and two days (D−2) for the a-FRR capacity price, and the system
operator's real-time (TEİAŞ) production series (`RT_Solar_Generation_Share`, `RT_Consumption`). Also tests whether
the solar effect is distinct from consumption (correlations, VIF, horse-race) and the net-load ramp mechanism.
Interpretive OLS with Newey–West (HAC, 24 lags) standard errors.

Produces: `Phase_11_2_Timing_Robustness.csv` and `Phase_11_2_Confounding_Diagnostics.csv` in `results/Phase_11_Robustness/`.

In [ ]:
# =============================================================================
# PHASE 11.2: TIMING, DECISION-TIME INFORMATION & CONFOUNDING (OLS-HAC) - CACHED
# =============================================================================
# Revision_Review_Referee_2.1 — Timing, decision-time info & confounding
# Added in revision. Feeds Supplementary Appendix J.1.
# =============================================================================
import pandas as pd, numpy as np, os, warnings
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
warnings.filterwarnings("ignore")

OUTPUT_DIR_P11R = os.path.join(RESULTS_BASE, "Phase_11_Robustness")
os.makedirs(OUTPUT_DIR_P11R, exist_ok=True)

# Core control set of the manuscript (natural gas excluded)
REV_CONTROLS = ['Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap',
                'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
                'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

# TEİAŞ real-time (ex-ante) series: RT solar share = solar MW / total generation MW, RT consumption = load MW
RT_SOLAR_SHARE_COL = 'RT_Solar_Generation_Share'
RT_CONSUMPTION_COL = 'RT_Consumption'

def ols_hac(data: pd.DataFrame, dep: str, xvars: list):
    """OLS of dep on xvars (+ constant) with Newey-West (HAC) standard errors, 24-hour lag window."""
    d = data.dropna(subset=[dep] + xvars)
    X = sm.add_constant(d[xvars].astype(float))
    return sm.OLS(d[dep].astype(float), X).fit(cov_type='HAC', cov_kwds={'maxlags': 24})

def run_timing_confounding(df: pd.DataFrame):
    print("\n>>> PHASE 11.2: TIMING / DECISION-TIME INFORMATION / CONFOUNDING (OLS-HAC)")
    path_t = os.path.join(OUTPUT_DIR_P11R, "Phase_11_2_Timing_Robustness.csv")
    path_c = os.path.join(OUTPUT_DIR_P11R, "Phase_11_2_Confounding_Diagnostics.csv")
    if os.path.exists(path_t) and os.path.exists(path_c):
        print(f"  [CACHE] Loaded from {path_t}")
        print(pd.read_csv(path_t).to_string(index=False))
        print(f"\n  [CACHE] Loaded from {path_c}")
        print(pd.read_csv(path_c).to_string(index=False)); return

    d = df.copy()   # hourly series, sorted by Date_Hour (Phase 1)
    d['solar_lag24'] = d['Solar_Generation_Share'].shift(24)   # day-ahead market: D-1
    d['solar_lag48'] = d['Solar_Generation_Share'].shift(48)   # a-FRR capacity: D-2
    d['Net_Load'] = d['Consumption'] - d['Solar_Generation']
    d['Net_Load_Ramp'] = d['Net_Load'].diff()
    cc = [c for c in REV_CONTROLS if c != 'Consumption']

    rows = []
    def add(block, label, dep, key, xvars):
        m = ols_hac(d, dep, xvars)
        rows.append({'Block': block, 'Specification': label, 'Dependent': dep, 'Regressor': key,
                     'Beta': m.params[key], 't_stat': m.tvalues[key], 'p_value': m.pvalues[key], 'N': int(m.nobs)})
    try:
        # [A] decision-time lags of the settlement (ex-post) solar share
        add('A: decision-time lag', 'MCP - realised, same hour', 'Mcp', 'Solar_Generation_Share', ['Solar_Generation_Share'] + REV_CONTROLS)
        add('A: decision-time lag', 'MCP - solar lagged one day (D-1)', 'Mcp', 'solar_lag24', ['solar_lag24'] + REV_CONTROLS)
        add('A: decision-time lag', 'a-FRR - realised, same hour', 'a-FRR_Price', 'Solar_Generation_Share', ['Solar_Generation_Share'] + REV_CONTROLS)
        add('A: decision-time lag', 'a-FRR - solar lagged two days (D-2)', 'a-FRR_Price', 'solar_lag48', ['solar_lag48'] + REV_CONTROLS)

        # [A'] real-time (ex-ante) TEİAŞ series; consumption also at decision time
        if RT_SOLAR_SHARE_COL in d.columns and RT_CONSUMPTION_COL in d.columns:
            d['rt_solar_lag24'] = d[RT_SOLAR_SHARE_COL].shift(24)
            d['rt_solar_lag48'] = d[RT_SOLAR_SHARE_COL].shift(48)
            d['rt_cons_lag24'] = d[RT_CONSUMPTION_COL].shift(24)
            d['rt_cons_lag48'] = d[RT_CONSUMPTION_COL].shift(48)
            add("A': real-time (ex-ante)", 'MCP - real-time solar (D-1) + real-time consumption', 'Mcp', 'rt_solar_lag24', ['rt_solar_lag24', RT_CONSUMPTION_COL] + cc)
            add("A': real-time (ex-ante)", 'MCP - fully ex-ante (real-time consumption, D-1)', 'Mcp', 'rt_solar_lag24', ['rt_solar_lag24', 'rt_cons_lag24'] + cc)
            add("A': real-time (ex-ante)", 'a-FRR - real-time solar (D-2) + real-time consumption', 'a-FRR_Price', 'rt_solar_lag48', ['rt_solar_lag48', RT_CONSUMPTION_COL] + cc)
            add("A': real-time (ex-ante)", 'a-FRR - fully ex-ante (real-time consumption, D-2)', 'a-FRR_Price', 'rt_solar_lag48', ['rt_solar_lag48', 'rt_cons_lag48'] + cc)
            for key in [RT_SOLAR_SHARE_COL, RT_CONSUMPTION_COL]:
                add("A': real-time horse-race", 'a-FRR ~ RT solar + RT consumption', 'a-FRR_Price', key, [RT_SOLAR_SHARE_COL, RT_CONSUMPTION_COL] + cc)
        else:
            print("  !! Real-time (TEİAŞ) columns not found -> block A' skipped.")

        # [B] horse-race: solar share and consumption in the same model
        for key in ['Solar_Generation_Share', 'Consumption']:
            add('B: horse-race', 'a-FRR ~ solar + consumption', 'a-FRR_Price', key, ['Solar_Generation_Share', 'Consumption'] + cc)

        # [C] mechanism: net load and its ramp
        add('C: net-load mechanism', 'a-FRR ~ net load', 'a-FRR_Price', 'Net_Load', ['Net_Load'] + cc)
        add('C: net-load mechanism', 'a-FRR ~ net-load ramp + solar', 'a-FRR_Price', 'Net_Load_Ramp', ['Net_Load_Ramp', 'Solar_Generation_Share'] + cc)

        timing = pd.DataFrame(rows)

        # Confounding diagnostics: correlations and VIF
        diag = []
        if RT_SOLAR_SHARE_COL in d.columns:
            diag.append({'Diagnostic': 'Correlation RT (ex-ante) vs settlement (ex-post) solar share', 'Item': 'all hours',
                         'Value': d[RT_SOLAR_SHARE_COL].corr(d['Solar_Generation_Share'])})
        diag.append({'Diagnostic': 'Correlation solar share vs consumption', 'Item': 'all hours',
                     'Value': d['Solar_Generation_Share'].corr(d['Consumption'])})
        for h in [10, 12, 14, 16, 17, 18, 19, 20]:
            sub = d[d['Hour'] == h]
            diag.append({'Diagnostic': 'Correlation solar share vs consumption', 'Item': f'hour {h}',
                         'Value': sub['Solar_Generation_Share'].corr(sub['Consumption'])})
        vifvars = ['Solar_Generation_Share', 'Consumption', 'Wind_Generation', 'Import_Coal_Generation',
                   'Hydroelectric_Generation', 'Domestic_Coal_Generation', 'Price_Cap']
        Xv = sm.add_constant(d[vifvars].dropna().astype(float))
        for i, v in enumerate(vifvars):
            diag.append({'Diagnostic': 'VIF', 'Item': v, 'Value': variance_inflation_factor(Xv.values, i + 1)})
        conf = pd.DataFrame(diag)

        timing.to_csv(path_t, index=False)
        conf.to_csv(path_c, index=False)
        print(timing.to_string(index=False))
        print("\n" + conf.to_string(index=False))
        print("\n  Note: OLS with Newey-West (HAC, 24 lags) standard errors; interpretive check, SARIMAX models unchanged.")
        print(f"  Saved: {path_t}\n  Saved: {path_c}")
    except Exception as e:
        print(f"  !! Error in timing/confounding checks: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_timing_confounding(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

### Phase 11.3: Sub-Period Parameter Stability (Revision_Review_Referee_2.2)

Re-estimates the solar coefficients separately for the earlier crisis/high-price period (2022–2023) and the later
high-solar period (2024–2025). Interpretive OLS with Newey–West (HAC, 24 lags) standard errors.

Produces: `Phase_11_3_SubPeriod.csv` in `results/Phase_11_Robustness/`.

In [ ]:
# =============================================================================
# PHASE 11.3: SUB-PERIOD PARAMETER STABILITY (OLS-HAC) - CACHED
# =============================================================================
# Revision_Review_Referee_2.2 — Sub-period (regime) stability
# Added in revision. Feeds Supplementary Appendix J.2.
# =============================================================================
import pandas as pd, os

def run_subperiod_stability(df: pd.DataFrame):
    print("\n>>> PHASE 11.3: SUB-PERIOD PARAMETER STABILITY (OLS-HAC)")
    path = os.path.join(OUTPUT_DIR_P11R, "Phase_11_3_SubPeriod.csv")
    if os.path.exists(path):
        print(f"  [CACHE] Loaded from {path}")
        print(pd.read_csv(path).to_string(index=False)); return

    years = pd.to_datetime(df['Date']).dt.year
    periods = [('Full sample', df),
               ('2022-2023', df[years <= 2023]),
               ('2024-2025', df[years >= 2024])]
    rows = []
    try:
        for dep, label in [('Mcp', 'MCP (merit-order)'), ('a-FRR_Price', 'a-FRR (balancing)')]:
            for pname, pdf in periods:
                m = ols_hac(pdf, dep, ['Solar_Generation_Share'] + REV_CONTROLS)
                rows.append({'Dependent': label, 'Period': pname,
                             'Beta': m.params['Solar_Generation_Share'],
                             't_stat': m.tvalues['Solar_Generation_Share'], 'N': int(m.nobs)})
        out = pd.DataFrame(rows)
        out.to_csv(path, index=False)
        print(out.to_string(index=False))
        print(f"\n  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in sub-period checks: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    target_df = locals().get('df', globals().get('df'))
    if target_df is not None:
        run_subperiod_stability(target_df)
    else:
        print("!! ERROR: DataFrame 'df' not found. Please run Phase 1.")

## Phase 10 (Appendix J): Additional Robustness Checks (Reviewer Response)

Assembles Supplementary **Appendix J** (Tables J1–J5) from the revision outputs of Phases 11.2, 11.3, 6.4, 8.2 and
7.4. Placed after Phase 11 because it reads their results; it only formats cached CSVs, so it is rebuilt on every
run. Outputs are written next to Appendices A–I in `results/Phase_10/`.

Produces: `Appendix_J1_Timing_Robustness.csv`, `Appendix_J2_SubPeriod.csv`, `Appendix_J3_OutOfSample.csv`,
`Appendix_J4_Threshold.csv`, `Appendix_J5_Clustered_QR.csv`.

In [ ]:
# =============================================================================
# PHASE 10 (APPENDIX J): SUPPLEMENTARY APPENDIX J AGGREGATION (J1-J5)
# =============================================================================
# Revision — Supplementary Appendix J aggregation
# Sources: Phase 11.2 (J1), Phase 11.3 (J2), Phase 6.4 (J3), Phase 8.2 (J4), Phase 7.4 (J5)
# =============================================================================
import pandas as pd, numpy as np, os

OUTPUT_DIR_P10 = os.path.join(RESULTS_BASE, "Phase_10")
os.makedirs(OUTPUT_DIR_P10, exist_ok=True)

J_SOURCES = {
    'J1': os.path.join(RESULTS_BASE, "Phase_11_Robustness", "Phase_11_2_Timing_Robustness.csv"),
    'J2': os.path.join(RESULTS_BASE, "Phase_11_Robustness", "Phase_11_3_SubPeriod.csv"),
    'J3': os.path.join(RESULTS_BASE, "Phase_6", "Phase_6_4_Out_of_Sample_CV.csv"),
    'J4': os.path.join(RESULTS_BASE, "Phase_8", "Phase_8_2_Hansen_Threshold.csv"),
    'J5': os.path.join(RESULTS_BASE, "Phase_7", "Phase_7_4_QR_Clustered_Bootstrap_SE.csv"),
}

def _bt(beta, t, stars=''):
    """Formats 'beta (t)' as in the Supplementary tables."""
    return f"{beta:,.0f} ({t:.1f}){stars}".replace('-', '−')

def generate_appendix_j():
    print("\n>>> PHASE 10 (APPENDIX J): ADDITIONAL ROBUSTNESS CHECKS (J1-J5)")
    missing = [k for k, p in J_SOURCES.items() if not os.path.exists(p)]
    if missing:
        print(f"  !! Missing sources for {missing}. Run Phases 6.4, 7.4, 8.2, 11.2 and 11.3 first."); return
    try:
        # J1 - timing / decision-time information (Phase 11.2, blocks A and A')
        t = pd.read_csv(J_SOURCES['J1'])
        order = ['MCP - realised, same hour', 'MCP - solar lagged one day (D-1)',
                 'MCP - real-time solar (D-1) + real-time consumption', 'MCP - fully ex-ante (real-time consumption, D-1)',
                 'a-FRR - realised, same hour', 'a-FRR - solar lagged two days (D-2)',
                 'a-FRR - real-time solar (D-2) + real-time consumption', 'a-FRR - fully ex-ante (real-time consumption, D-2)']
        t = t[t['Block'].str.startswith(('A:', "A':")) & t['Specification'].isin(order)]
        j1 = t.set_index('Specification').loc[order, ['Beta', 't_stat', 'p_value']].round(4).reset_index()
        j1.columns = ['Specification', 'Beta (solar)', 't-stat', 'p-value']

        # J2 - sub-period stability (Phase 11.3)
        s = pd.read_csv(J_SOURCES['J2'])
        j2 = (s.assign(Cell=[_bt(b, tt) for b, tt in zip(s['Beta'], s['t_stat'])])
               .pivot(index='Dependent', columns='Period', values='Cell')[['Full sample', '2022-2023', '2024-2025']]
               .reset_index().rename(columns={'Dependent': 'Coefficient (t-stat)'}))

        # J3 - out-of-sample validation (Phase 6.4)
        o = pd.read_csv(J_SOURCES['J3'])
        fit = []
        for _, r in o.iterrows():
            if pd.notna(r['R2_oos']):
                fit.append(f"R² = {r['R2_oos']:.3f}")
            else:
                extra = (f"; ROC-AUC = {r['ROC_AUC']:.3f}; balanced acc = {r['Balanced_Acc']:.3f}; majority-class acc = {r['Majority_baseline_acc']:.3f}"
                         if 'ROC_AUC' in o.columns and pd.notna(r.get('ROC_AUC', np.nan)) else '')
                fit.append(f"Acc = {r['Accuracy']:.3f}; F1 = {r['F1']:.3f}" + extra)
        fit_in = [f"R² = {r:.3f}" if pd.notna(r) else f"Acc = {a:.3f}"
                  for r, a in zip(o.get('R2_insample', [np.nan] * len(o)), o.get('Accuracy_insample', [np.nan] * len(o)))]
        j3 = pd.DataFrame({'Target': o['Target'], 'MAE': o['MAE'].round(0), 'RMSE': o['RMSE'].round(0),
                           'Held-out fit': fit, 'In-sample fit (full-sample model)': fit_in})

        # J4 - threshold identification (Phase 8.2)
        h = pd.read_csv(J_SOURCES['J4']).iloc[0]
        p_txt = (f"p < {1 / h['N_Boot_supF']:.3f}" if h['supF_pval_bootstrap'] == 0 else f"p = {h['supF_pval_bootstrap']:.3f}")
        j4 = pd.DataFrame({'Quantity': ['Model',
                                        'Onset threshold γ (point estimate)', '90% bootstrap confidence interval for γ',
                                        'Slope change at the kink δ (TL/MW per percentage point)',
                                        'Test of no kink (H0: δ = 0), sup-F',
                                        'Upper crossover (surplus overtakes deficit)'],
                           'Estimate': ['Continuous regression kink (Hansen, 2017), least-squares grid 5–44%',
                                        f"{h['Threshold_Point_Estimate_pct']:.0f}% solar share",
                                        f"[{h['CI90_Lower_pct']:.0f}%, {h['CI90_Upper_pct']:.0f}%] (median {h['Bootstrap_Median_pct']:.0f}%; day-clustered bootstrap, {int(h['N_Boot'])} reps)",
                                        (f"{h['delta']:,.1f}".replace('-', '−') + f" (day-clustered bootstrap SE {h['delta_se']:,.1f}; t = {h['delta_t']:.1f}); slope "
                                         + f"{h['slope_pre_kink_b']:,.1f}".replace('-', '−') + " below vs " + f"{h['slope_post_kink_b_plus_delta']:,.1f}".replace('-', '−') + " above γ"),
                                        f"F = {h['supF_stat']:.1f}; day-clustered wild-bootstrap {p_txt} ({int(h['N_Boot_supF'])} reps)",
                                        '~33% (XGBoost interaction simulation, Phase 6.2; descriptive, no confidence interval)']})

        # J5 - dependence-robust quantile regression (Phase 7.4)
        q = pd.read_csv(J_SOURCES['J5'])
        q['Cell'] = [_bt(b, tt, s if isinstance(s, str) else '')
                     for b, tt, s in zip(q['Coef'], q['t_stat'], q['Sig'])]
        q['Col'] = 'Q' + (q['Quantile'] * 100).round().astype(int).astype(str)
        j5 = (q.pivot(index='Hypothesis', columns='Col', values='Cell')[['Q10', 'Q50', 'Q90']]
               .reindex(['MCP', 'aFRR']).rename(index={'MCP': 'MCP β (t)', 'aFRR': 'a-FRR β (t)'})
               .reset_index().rename(columns={'Hypothesis': 'Dependent variable'}))

        for name, tab, title in [('Appendix_J1_Timing_Robustness', j1, 'J1. Timing, decision-time information'),
                                 ('Appendix_J2_SubPeriod', j2, 'J2. Sub-period parameter stability'),
                                 ('Appendix_J3_OutOfSample', j3, 'J3. Out-of-sample validation'),
                                 ('Appendix_J4_Threshold', j4, 'J4. Threshold identification'),
                                 ('Appendix_J5_Clustered_QR', j5, 'J5. Dependence-robust quantile regression (*** p<0.01)')]:
            path = os.path.join(OUTPUT_DIR_P10, name + ".csv")
            tab.to_csv(path, index=False, encoding='utf-8-sig', **CSV_KW)
            print(f"\n  Table {title}\n" + tab.to_string(index=False))
            print(f"  Saved: {path}")
    except Exception as e:
        print(f"  !! Error in Appendix J aggregation: {e}")

# === MAIN EXECUTION ===
if __name__ == "__main__":
    generate_appendix_j()

## Submission Figures — dergi numaralandırmasıyla dışa aktarma

Aşağıdaki hücre, makaledeki figür numaralarına göre 10 PNG'yi `results/Submission_Figures/` klasörüne kopyalar
(`Figure_1a.png` … `Figure_9.png`) ve her biri için piksel boyutu, dpi etiketi ve 19 cm çift-sütun baskıdaki
etkin çözünürlüğü raporlar. Editorial Manager'a **bu klasördeki dosyalar** yüklenir.


In [ ]:
# =============================================================================
# SUBMISSION FIGURES: MANUSCRIPT-NUMBERED EXPORT (Figure_1a ... Figure_9)
# =============================================================================
import glob, shutil
from PIL import Image

SUBMISSION_DIR = os.path.join(RESULTS_BASE, "Submission_Figures")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# (hedef ad, kaynak desen) -- makale numarasi <- pipeline dosyasi
_submission_map = [
    ("Figure_1a", os.path.join("Phase_5", "Phase_5_Figure_1_*.png")),
    ("Figure_1b", os.path.join("Phase_5", "Phase_5_Figure_2_*.png")),
    ("Figure_2",  os.path.join("Phase_9", "Figure_3_*.png")),
    ("Figure_3",  os.path.join("Phase_5", "Phase_5_Figure_4_*.png")),
    ("Figure_4",  os.path.join("Phase_6", "Phase_6_Figure_5_*.png")),
    ("Figure_5",  os.path.join("Phase_5", "Phase_5_Figure_6_*.png")),
    ("Figure_6",  os.path.join("Phase_6", "Phase_6_Figure_7_*.png")),
    ("Figure_7",  os.path.join("Phase_6", "Phase_6_Figure_8_*.png")),
    ("Figure_8",  os.path.join("Phase_6", "Phase_6_Figure_9_*.png")),
    ("Figure_9",  os.path.join("Phase_8", "Phase_8_Figure_10_GAM_Flexibility_Stress.png")),
]

print(f">>> SUBMISSION FIGURES -> {SUBMISSION_DIR}\n")
_problems = []
for _target, _pattern in _submission_map:
    _hits = sorted(glob.glob(os.path.join(RESULTS_BASE, _pattern)))
    if len(_hits) != 1:
        _problems.append(f"{_target}: desen '{_pattern}' icin {len(_hits)} eslesme bulundu {_hits}")
        continue
    _dst = os.path.join(SUBMISSION_DIR, _target + ".png")
    shutil.copy2(_hits[0], _dst)
    with Image.open(_dst) as _im:
        _w, _h = _im.size
        _tag = _im.info.get("dpi", ("?", "?"))[0]
    _eff = _w / (19.0 / 2.54)   # 19 cm cift-sutun baskida etkin dpi
    print(f"  {_target:<10} {_w}x{_h}px | dpi etiketi: {_tag} | 19 cm baskida ~{_eff:.0f} dpi  <- {os.path.basename(_hits[0])}")

if _problems:
    print("\n!! SORUN:")
    for _p in _problems: print("   -", _p)
else:
    print("\n>> 10/10 figur hazir. EM'e 'results/Submission_Figures/' icindekileri yukleyin.")


## Supplementary Figures — ek dosya figürlerinin dışa aktarımı

Supplementary Material içinde gömülü beş figürü (F.1, F.2, H1, I.1, I.2) kendi etiketleriyle
`results/Supplementary_Figures/` klasörüne kopyalar ve çözünürlük raporu basar.
**Not:** Bunlar EM'e ayrı 'Figure' kalemi olarak YÜKLENMEZ — Supplementary docx'in içinde
gömülü giderler; bu klasör docx görsellerini tazelemek ve repo bütünlüğü içindir.


In [ ]:
# =============================================================================
# SUPPLEMENTARY FIGURES: LABELLED EXPORT (F1, F2, H1, I1, I2)
# =============================================================================
import shutil
from PIL import Image

SUPP_FIG_DIR = os.path.join(RESULTS_BASE, "Supplementary_Figures")
os.makedirs(SUPP_FIG_DIR, exist_ok=True)

# (hedef ad, kaynak dosya) -- supplementary etiketi <- pipeline ciktisi (tam ad, joker yok)
_supp_map = [
    ("Supplementary_Figure_F1", os.path.join("Phase_10", "Appendix_F1_MCP_Correlation.png")),
    ("Supplementary_Figure_F2", os.path.join("Phase_10", "Appendix_F2_aFRR_Correlation.png")),
    ("Supplementary_Figure_H1", os.path.join("Phase_10", "Appendix_H_Down_Regulation_Volume.png")),
    ("Supplementary_Figure_I1", os.path.join("Phase_10", "Appendix_I_Figure_I1_Flexibility_Trap_Botas.png")),
    ("Supplementary_Figure_I2", os.path.join("Phase_10", "Appendix_I_Figure_I2_GAM_Flexibility_Stress_Botas.png")),
]

print(f">>> SUPPLEMENTARY FIGURES -> {SUPP_FIG_DIR}\n")
_missing = []
for _target, _rel in _supp_map:
    _srcp = os.path.join(RESULTS_BASE, _rel)
    if not os.path.exists(_srcp):
        _missing.append(f"{_target}: kaynak yok -> {_srcp}")
        continue
    _dst = os.path.join(SUPP_FIG_DIR, _target + ".png")
    shutil.copy2(_srcp, _dst)
    with Image.open(_dst) as _im:
        _w, _h = _im.size
        _tag = _im.info.get("dpi", ("?", "?"))[0]
    print(f"  {_target:<26} {_w}x{_h}px | dpi etiketi: {_tag}  <- {os.path.basename(_srcp)}")

if _missing:
    print("\n!! SORUN:")
    for _m in _missing: print("   -", _m)
else:
    print("\n>> 5/5 supplementary figur kopyalandi.")
    print(">> Bunlar Supplementary docx icinde gomulu gider; EM'e ayri Figure kalemi olarak yuklenmez.")
